### 📈 Quant Explains the "Day Trader" Scam

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [🛡️ Apply to Run a Personal Hedge Fund](https://quantguild.com/personal-hedge-fund)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

##### **Problem:** The Lottery Ticket Effect

Most don't know what **good** is in the context of a return, let alone a risk-adjusted return

Is it normal to have *20%, 50%, 150%, 1,000%* returns?  How much risk did you have to take to generate that return.

**Day Trader Premise**

I have indicators that can produce an outsized return, and I can teach it to you or if you give me money I can run the strategy

It's not about if they're right or wrong (which they are), or that "if they had a profitable strategy they wouldn't share it" that's such bullshit, here's a profitable strategy: buy SPY.

The problem is the fact that you don't know what risk you're exposed to...

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

START_DATE = "2025-01-01"
END_DATE = "2026-12-31"

SEED = 42
INITIAL_WEALTH = 100

LEFT_PATH_NAME = "Day Trading Student"
RIGHT_PATH_NAME = "Positive EV Strategy"

# Day Trading Student settings
STUDENT_PRE_BLOWOUT_FRACTION = 0.78
STUDENT_DAILY_LOG_DRIFT = 0.0002
STUDENT_DAILY_LOG_VOL = 0.075
STUDENT_NUM_MANIA_SHOCKS = 20
STUDENT_SHOCKS = [0.18, 0.24, 0.30, -0.18, -0.24, -0.30]
BLOWOUT_REMAINING_CAPITAL = 0.06
POST_BLOWOUT_DECAY = 1.70
POST_BLOWOUT_NOISE_VOL = 0.035

# Positive EV settings
EV_ANNUAL_EDGE = 0.16
EV_DAILY_LOG_VOL = 0.0035
TARGET_EV_TOTAL_RETURN = 0.34

# Animation settings
FRAME_STRIDE = 5
FRAME_DURATION = 20

# ============================================================
# Generate synthetic trading dates
# ============================================================

dates = pd.bdate_range(START_DATE, END_DATE)

if len(dates) < 2:
    raise ValueError("Need at least two business days to create an animated wealth path.")

rng = np.random.default_rng(SEED)
n_days = len(dates)

# ============================================================
# Generate Day Trading Student wealth path
# ============================================================

blowout_idx = int(n_days * STUDENT_PRE_BLOWOUT_FRACTION)
blowout_idx = max(30, min(blowout_idx, n_days - 2))

student_pre_log_returns = rng.normal(
    loc=STUDENT_DAILY_LOG_DRIFT,
    scale=STUDENT_DAILY_LOG_VOL,
    size=blowout_idx
)
student_pre_log_returns[0] = 0

# Inject oversized wins/losses to create manic swings before the blowout.
shock_pool = np.arange(10, blowout_idx - 10)

if len(shock_pool) > 0:
    n_shocks = min(STUDENT_NUM_MANIA_SHOCKS, len(shock_pool))
    shock_indices = rng.choice(shock_pool, size=n_shocks, replace=False)
    shock_values = rng.choice(STUDENT_SHOCKS, size=n_shocks, replace=True)
    student_pre_log_returns[shock_indices] += shock_values

student_pre_wealth = INITIAL_WEALTH * np.exp(np.cumsum(student_pre_log_returns))

# Blowout: sudden loss of most remaining capital, followed by decay near zero.
post_blowout_n = n_days - blowout_idx
post_blowout_start = max(student_pre_wealth[-1] * BLOWOUT_REMAINING_CAPITAL, 1.0)

post_blowout_decay_curve = np.exp(
    -np.linspace(0, POST_BLOWOUT_DECAY, post_blowout_n)
)

post_blowout_noise = np.clip(
    1 + rng.normal(0, POST_BLOWOUT_NOISE_VOL, post_blowout_n),
    0.90,
    1.10
)

student_post_wealth = (
    post_blowout_start
    * post_blowout_decay_curve
    * post_blowout_noise
)
student_post_wealth[0] = post_blowout_start

student_wealth = np.concatenate([student_pre_wealth, student_post_wealth])

# ============================================================
# Generate positive EV, low-variance wealth path
# ============================================================

ev_daily_log_mean = np.log1p(EV_ANNUAL_EDGE) / 252

ev_log_returns = rng.normal(
    loc=ev_daily_log_mean,
    scale=EV_DAILY_LOG_VOL,
    size=n_days
)
ev_log_returns[0] = 0

# Keep the path stochastic but pin the final outcome to a clean positive-EV target.
target_ev_total_log_return = np.log1p(TARGET_EV_TOTAL_RETURN)

ev_alpha_adjustment_per_day = (
    target_ev_total_log_return - ev_log_returns.sum()
) / (n_days - 1)

ev_log_returns[1:] += ev_alpha_adjustment_per_day

ev_wealth = INITIAL_WEALTH * np.exp(np.cumsum(ev_log_returns))

# ============================================================
# Build DataFrame
# ============================================================

df = pd.DataFrame({
    "date": dates,
    "student_wealth": student_wealth,
    "ev_wealth": ev_wealth
})

df["student_simple_return"] = df["student_wealth"].pct_change().fillna(0)
df["ev_simple_return"] = df["ev_wealth"].pct_change().fillna(0)

df["student_cum_return"] = df["student_wealth"] / INITIAL_WEALTH - 1
df["ev_cum_return"] = df["ev_wealth"] / INITIAL_WEALTH - 1

# ============================================================
# Summary stats
# ============================================================

latest_date = df["date"].iloc[-1]

student_final_return = df["student_cum_return"].iloc[-1]
ev_final_return = df["ev_cum_return"].iloc[-1]
return_spread = ev_final_return - student_final_return

def annualized_cagr(final_wealth, initial_wealth, n_days):
    years = n_days / 252
    return (final_wealth / initial_wealth) ** (1 / years) - 1

def annualized_sharpe(simple_returns):
    simple_returns = np.asarray(simple_returns)

    if len(simple_returns) < 2:
        return np.nan

    mean = np.mean(simple_returns)
    std = np.std(simple_returns, ddof=1)

    if std == 0:
        return np.nan

    return mean / std * np.sqrt(252)

def max_drawdown_from_wealth(wealth):
    wealth = np.asarray(wealth)
    running_max = np.maximum.accumulate(wealth)
    drawdown = 1 - wealth / running_max
    return np.max(drawdown)

student_cagr = annualized_cagr(
    df["student_wealth"].iloc[-1],
    INITIAL_WEALTH,
    len(df)
)

ev_cagr = annualized_cagr(
    df["ev_wealth"].iloc[-1],
    INITIAL_WEALTH,
    len(df)
)

student_sharpe = annualized_sharpe(df["student_simple_return"])
ev_sharpe = annualized_sharpe(df["ev_simple_return"])

student_mdd = max_drawdown_from_wealth(df["student_wealth"])
ev_mdd = max_drawdown_from_wealth(df["ev_wealth"])

student_peak = df["student_wealth"].max()
student_final_wealth = df["student_wealth"].iloc[-1]
ev_final_wealth = df["ev_wealth"].iloc[-1]

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"
student_color = "#ffaa33"
ev_color = "#00ff88"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# Shared y-axis range across both charts
wealth_min = min(df["student_wealth"].min(), df["ev_wealth"].min())
wealth_max = max(df["student_wealth"].max(), df["ev_wealth"].max())

y_pad = max((wealth_max - wealth_min) * 0.12, 5)
y_lower = max(0, wealth_min - y_pad)
y_upper = wealth_max + y_pad

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(
        f"{LEFT_PATH_NAME}: CAGR {student_cagr:.2%} | "
        f"Sharpe {student_sharpe:.2f} | "
        f"MDD {student_mdd:.1%}",

        f"{RIGHT_PATH_NAME}: CAGR {ev_cagr:.2%} | "
        f"Sharpe {ev_sharpe:.2f} | "
        f"MDD {ev_mdd:.1%}"
    )
)

# Starting wealth baselines
fig.add_hline(
    y=INITIAL_WEALTH,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1
)

fig.add_hline(
    y=INITIAL_WEALTH,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2
)

# Left: Day Trading Student wealth path
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["student_wealth"].iloc[0]],
        mode="lines",
        line=dict(color=student_color, width=4),
        name=LEFT_PATH_NAME,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{LEFT_PATH_NAME} Wealth: $%{{y:.2f}}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# Right: Positive EV wealth path
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["ev_wealth"].iloc[0]],
        mode="lines",
        line=dict(color=ev_color, width=4),
        name=RIGHT_PATH_NAME,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{RIGHT_PATH_NAME} Wealth: $%{{y:.2f}}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(1, len(df), FRAME_STRIDE))

if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    plot_slice = df.iloc[:i + 1]

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=plot_slice["date"],
                    y=plot_slice["student_wealth"]
                ),
                go.Scatter(
                    x=plot_slice["date"],
                    y=plot_slice["ev_wealth"]
                )
            ],
            traces=[0, 1],
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["date"].iloc[i].strftime("%b %Y"),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "Day Trading Path vs. Positive EV Wealth Path"
            "<br><sup>"
            f"{START_DATE} to {latest_date:%Y-%m-%d} · "
            f"{LEFT_PATH_NAME} Final Wealth: ${student_final_wealth:.2f} · "
            f"Peak Before Blowout: ${student_peak:.2f} · "
            f"{RIGHT_PATH_NAME} Final Wealth: ${ev_final_wealth:.2f} · "
            f"Return Spread: {return_spread:.1%}"
            "</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1200,
    margin=dict(t=120, b=150, r=50, l=75),
    showlegend=False,
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": False
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {
                            "duration": 0,
                            "redraw": False
                        },
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_annotations(font=dict(color=off_white, size=14))

# Left subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[y_lower, y_upper],
    title_text="Portfolio Value, Starting at $100",
    tickprefix="$",
    tickformat=",.0f"
)

# Right subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[90, 150],
    title_text="Portfolio Value, Starting at $100",
    tickprefix="$",
    tickformat=",.0f"
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("student_vs_positive_ev_animated.html", include_plotlyjs="cdn")

###### ______________________________________________________________________________________________________________________________________

##### **Reality:** Winners are the Casino, not the Player

Harvesting VRP, engineering a hedged portfolio, etc...all unsexy, they all take time to generate returns, just like being the house.

The space is statistical *at best*, the only way to make money in the long run is to be the house, like a casino

There is no such thing as *hitting a big trade*, institutional managers would laugh you out of the room if you bet the house on black

It's about positioning and survival, but everyone wants a quick hit, so the day traders win this round against reality...

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

START_DATE = "2025-01-01"
END_DATE = "2026-12-31"

SEED = 42
INITIAL_WEALTH = 100

LEFT_PATH_NAME = "Casino: Other Side of Player Bets"
RIGHT_PATH_NAME = "Roulette Player Bankrolls"

PLAYER_PATH_NAME = "Player"

# More players makes the casino side look statistically boring
# because idiosyncratic player luck diversifies away.
NUM_PLAYERS = 75

# Animation settings
FRAME_STRIDE = 5
FRAME_DURATION = 20

# Figure settings
FIG_WIDTH = 1200
FIG_HEIGHT = 650
HORIZONTAL_SPACING = 0.12

# Y-axis ranges
CASINO_Y_RANGE = [90, 205]
PLAYER_Y_RANGE = [0, 160]

# Player path opacity
PLAYER_PATH_OPACITY = 0.30

# Roulette settings
ROULETTE_WIN_PROBABILITY = 18 / 38

# Player behavior settings
ROULETTE_BASE_BET_LOW = 0.025
ROULETTE_BASE_BET_HIGH = 0.070
ROULETTE_MAX_BET_FRACTION = 0.48

# Every player eventually blows out
BLOWOUT_START_FRACTION = 0.35
BLOWOUT_END_FRACTION = 0.95

# ============================================================
# Generate synthetic trading dates
# ============================================================

dates = pd.bdate_range(START_DATE, END_DATE)

if len(dates) < 2:
    raise ValueError("Need at least two business days to create an animated wealth path.")

rng = np.random.default_rng(SEED)
n_days = len(dates)

# ============================================================
# Helper functions
# ============================================================

def annualized_cagr(final_wealth, initial_wealth, n_days):
    years = n_days / 252
    return (final_wealth / initial_wealth) ** (1 / years) - 1


def max_drawdown_from_wealth(wealth):
    wealth = np.asarray(wealth, dtype=float)
    running_max = np.maximum.accumulate(wealth)
    drawdown = 1 - wealth / running_max
    return np.max(drawdown)


def generate_roulette_player_paths(
    rng,
    n_paths,
    n_days,
    initial_wealth=100
):
    """
    Generates player bankroll paths.

    Each player is betting against the casino.
    Player bankroll changes are later used directly to compute casino P&L.

    Every player eventually blows out.
    """
    player_paths = np.empty((n_paths, n_days))

    min_blowout_idx = int(n_days * BLOWOUT_START_FRACTION)
    max_blowout_idx = int(n_days * BLOWOUT_END_FRACTION)

    min_blowout_idx = max(20, min_blowout_idx)
    max_blowout_idx = min(n_days - 2, max_blowout_idx)

    for player_idx in range(n_paths):
        wealth = np.empty(n_days)
        wealth[0] = initial_wealth

        blowout_idx = int(rng.integers(min_blowout_idx, max_blowout_idx + 1))

        base_bet_fraction = rng.uniform(
            ROULETTE_BASE_BET_LOW,
            ROULETTE_BASE_BET_HIGH
        )

        loss_streak = 0

        # Normal gambling phase
        for t in range(1, blowout_idx):
            # Players increase bet size after losses,
            # creating occasional rallies but eventually fragile bankrolls.
            stake_fraction = base_bet_fraction * (1 + 0.40 * loss_streak)
            stake_fraction = min(stake_fraction, ROULETTE_MAX_BET_FRACTION)

            stake = wealth[t - 1] * stake_fraction

            if rng.random() < ROULETTE_WIN_PROBABILITY:
                # Player wins an even-money roulette-style bet.
                wealth[t] = wealth[t - 1] + stake
                loss_streak = 0
            else:
                # Player loses stake to the casino.
                wealth[t] = wealth[t - 1] - stake
                loss_streak += 1

            wealth[t] = np.clip(wealth[t], 0.10, 160)

        # Blowout phase:
        # the remaining bankroll is lost to the casino over time.
        post_blowout_n = n_days - blowout_idx

        blowout_start = max(
            wealth[blowout_idx - 1] * rng.uniform(0.025, 0.085),
            0.10
        )

        final_floor = rng.uniform(0.05, 1.25)

        decay_speed = rng.uniform(2.8, 4.8)
        decay_curve = np.exp(-np.linspace(0, decay_speed, post_blowout_n))

        post_noise = np.clip(
            1 + rng.normal(0, 0.025, post_blowout_n),
            0.92,
            1.08
        )

        post_blowout_wealth = (
            final_floor
            + (blowout_start - final_floor) * decay_curve * post_noise
        )

        post_blowout_wealth[0] = blowout_start
        post_blowout_wealth[-1] = final_floor

        wealth[blowout_idx:] = np.maximum(post_blowout_wealth, 0.05)

        # Force final blowout.
        wealth[-1] = final_floor

        player_paths[player_idx] = wealth

    return player_paths


def build_casino_other_side_path(
    player_paths,
    initial_wealth=100
):
    """
    Casino value is literally the other side of the player pool.

    If the average player loses $1, the casino gains $1.
    If the average player wins $1, the casino loses $1.

    This is not smoothed, not fitted, and not independently generated.
    """
    average_player_wealth = player_paths.mean(axis=0)
    average_player_pnl = average_player_wealth - initial_wealth

    casino_value = initial_wealth - average_player_pnl

    return casino_value


# ============================================================
# Generate player paths
# ============================================================

player_paths = generate_roulette_player_paths(
    rng=rng,
    n_paths=NUM_PLAYERS,
    n_days=n_days,
    initial_wealth=INITIAL_WEALTH
)

# ============================================================
# Casino path = exact other side of average player P&L
# ============================================================

casino_wealth = build_casino_other_side_path(
    player_paths=player_paths,
    initial_wealth=INITIAL_WEALTH
)

# ============================================================
# Build DataFrame
# ============================================================

df = pd.DataFrame({
    "date": dates,
    "casino_wealth": casino_wealth,
    "average_player_wealth": player_paths.mean(axis=0)
})

df["casino_simple_return"] = df["casino_wealth"].pct_change().fillna(0)
df["casino_cum_return"] = df["casino_wealth"] / INITIAL_WEALTH - 1

# ============================================================
# Summary stats
# ============================================================

latest_date = df["date"].iloc[-1]

casino_final_wealth = df["casino_wealth"].iloc[-1]
casino_final_return = df["casino_cum_return"].iloc[-1]

casino_cagr = annualized_cagr(
    final_wealth=casino_final_wealth,
    initial_wealth=INITIAL_WEALTH,
    n_days=len(df)
)

casino_mdd = max_drawdown_from_wealth(df["casino_wealth"])

average_player_final_wealth = player_paths[:, -1].mean()
median_player_final_wealth = np.median(player_paths[:, -1])
players_blown_out = np.sum(player_paths[:, -1] <= 2.00)

total_player_starting_wealth = NUM_PLAYERS * INITIAL_WEALTH
total_player_final_wealth = player_paths[:, -1].sum()
total_player_loss = total_player_starting_wealth - total_player_final_wealth

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

casino_color = "#ffaa33"
player_color = "rgb(255, 55, 55)"
average_player_color = "#ffaaaa"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
    automargin=True
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=HORIZONTAL_SPACING,
    subplot_titles=(
        f"{LEFT_PATH_NAME}: CAGR {casino_cagr:.2%} | MDD {casino_mdd:.1%}",
        f"{RIGHT_PATH_NAME}: {players_blown_out}/{NUM_PLAYERS} Blown Out"
    )
)

# Starting wealth baselines
fig.add_hline(
    y=INITIAL_WEALTH,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1
)

fig.add_hline(
    y=INITIAL_WEALTH,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2
)

# ============================================================
# Initial traces
# ============================================================

# Left: casino path, exact inverse of average player P&L
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["casino_wealth"].iloc[0]],
        mode="lines",
        line=dict(color=casino_color, width=4.75),
        name=LEFT_PATH_NAME,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Casino Value: $%{y:.2f}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

casino_trace_index = 0
first_player_trace_index = 1

# Right: individual player bankroll paths
for player_idx in range(NUM_PLAYERS):
    fig.add_trace(
        go.Scatter(
            x=[df["date"].iloc[0]],
            y=[player_paths[player_idx, 0]],
            mode="lines",
            line=dict(color=player_color, width=1.10),
            opacity=PLAYER_PATH_OPACITY,
            name=f"{PLAYER_PATH_NAME} {player_idx + 1}",
            hoverinfo="skip"
        ),
        row=1,
        col=2
    )

average_player_trace_index = 1 + NUM_PLAYERS

# Right: average player bankroll path, so the casino inverse is visually checkable
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["average_player_wealth"].iloc[0]],
        mode="lines",
        line=dict(color=average_player_color, width=3.25),
        name="Average Player",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Average Player Wealth: $%{y:.2f}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

total_traces = 1 + NUM_PLAYERS + 1

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(1, len(df), FRAME_STRIDE))

if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    plot_dates = df["date"].iloc[:i + 1]

    frame_data = []

    # Left casino path
    frame_data.append(
        go.Scatter(
            x=plot_dates,
            y=df["casino_wealth"].iloc[:i + 1]
        )
    )

    # Right player paths
    for player_idx in range(NUM_PLAYERS):
        frame_data.append(
            go.Scatter(
                x=plot_dates,
                y=player_paths[player_idx, :i + 1]
            )
        )

    # Right average player path
    frame_data.append(
        go.Scatter(
            x=plot_dates,
            y=df["average_player_wealth"].iloc[:i + 1]
        )
    )

    frames.append(
        go.Frame(
            data=frame_data,
            traces=list(range(total_traces)),
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["date"].iloc[i].strftime("%b %Y"),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "Casino as the Other Side of Roulette Player Bets"
            "<br><sup>"
            f"{START_DATE} to {latest_date:%Y-%m-%d} · "
            f"Casino Final Value: ${casino_final_wealth:.2f} · "
            f"Casino Return: {casino_final_return:.1%} · "
            f"Avg Player Final Wealth: ${average_player_final_wealth:.2f} · "
            f"Median Player Final Wealth: ${median_player_final_wealth:.2f} · "
            f"Total Player Loss: ${total_player_loss:,.2f} · "
            f"Players Blown Out: {players_blown_out}/{NUM_PLAYERS}"
            "</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=FIG_HEIGHT,
    width=FIG_WIDTH,
    margin=dict(t=120, b=150, r=55, l=75),
    showlegend=False,
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": False
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {
                            "duration": 0,
                            "redraw": False
                        },
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_annotations(font=dict(color=off_white, size=14))

# ============================================================
# Axes
# ============================================================

# Left subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=CASINO_Y_RANGE,
    title_text="Casino Value per $100 Player Stake",
    tickprefix="$",
    tickformat=",.0f"
)

# Right subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=PLAYER_Y_RANGE,
    title_text="Player Bankroll, Starting at $100",
    tickprefix="$",
    tickformat=",.0f"
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("casino_other_side_vs_player_blowouts_animated.html", include_plotlyjs="cdn")

###### ______________________________________________________________________________________________________________________________________

##### **Problem:** The Environment is Statistical (*at Best*)

The entire space is statistical, at best.  

The randomness in practice does not converge, and to make matters worse, it's randomness indexed *over time*.

In reality, we don't have nice classroom convergence of conditional and unconditional probabilities, statistics, and distributions.

This makes it literally impossible to determine if someone's advice is *doing anything* as we lack counterfactuals.

This is a big problem in the finance literature in general.


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

START_DATE = "2025-01-01"
END_DATE = "2026-12-31"

SEED = 42
INITIAL_PRICE = 100

LEFT_PATH_NAME = "Observed Market"
RIGHT_PATH_NAME = "Counterfactual Market"

BUY_INDICATOR_LABEL = "Buy Indicator"
BUY_INDICATOR_LABEL_LEFT = "Buy Indicator"
BUY_INDICATOR_LABEL_RIGHT = "Buy Indicator ?"

# Timeline settings
BUY_INDICATOR_FRACTION = 0.42
GAP_DELAY_DAYS = 12

# Observed path behavior
PRE_EVENT_DAILY_NOISE = 0.08
POST_EVENT_DAILY_NOISE = 0.12

GAP_UP_PCT = 0.055
POST_GAP_TOTAL_TREND_PCT = 0.22

# Counterfactual path behavior
COUNTERFACTUAL_PRE_NOISE = 0.06
COUNTERFACTUAL_POST_NOISE = 0.08
COUNTERFACTUAL_FINAL_PRICE = 91

# Animation settings
FRAME_STRIDE = 5
FRAME_DURATION = 20

# Figure settings
FIG_WIDTH = 1200
FIG_HEIGHT = 650

LEFT_Y_RANGE = [95, 135]
RIGHT_Y_RANGE = [85, 105]

# ============================================================
# Generate synthetic trading dates
# ============================================================

dates = pd.bdate_range(START_DATE, END_DATE)

if len(dates) < 2:
    raise ValueError("Need at least two business days to create an animated path.")

rng = np.random.default_rng(SEED)
n_days = len(dates)

buy_indicator_idx = int(n_days * BUY_INDICATOR_FRACTION)
buy_indicator_idx = max(20, min(buy_indicator_idx, n_days - GAP_DELAY_DAYS - 20))

gap_idx = buy_indicator_idx + GAP_DELAY_DAYS

buy_indicator_date = dates[buy_indicator_idx]
gap_date = dates[gap_idx]

# Convert pandas Timestamp to native Python datetime for Plotly shapes/annotations.
buy_indicator_x = pd.Timestamp(buy_indicator_date).to_pydatetime()

# ============================================================
# Generate observed path: horizontal, indicator, gap up, uptrend
# ============================================================

def build_gap_up_trend_path(
    rng,
    n_days,
    initial_price=100,
    gap_idx=112,
    gap_up_pct=0.055,
    post_gap_total_trend_pct=0.22,
    pre_event_daily_noise=0.08,
    post_event_daily_noise=0.12
):
    """
    Builds the observed path:

    1. Horizontal / range-bound market.
    2. Buy indicator appears before the move.
    3. Gap up.
    4. Persistent uptrend.
    """
    price = np.empty(n_days)
    price[0] = initial_price

    # Pre-gap horizontal market
    for i in range(1, gap_idx):
        mean_reversion = 0.040 * (initial_price - price[i - 1])
        noise = rng.normal(0, pre_event_daily_noise)
        price[i] = price[i - 1] + mean_reversion + noise

    # Gap up
    price[gap_idx] = price[gap_idx - 1] * (1 + gap_up_pct)

    # Post-gap uptrend
    post_gap_n = n_days - gap_idx - 1

    if post_gap_n > 0:
        target_final_price = price[gap_idx] * (1 + post_gap_total_trend_pct)

        trend_line = np.linspace(
            price[gap_idx],
            target_final_price,
            post_gap_n + 1
        )

        trend_noise = np.cumsum(
            rng.normal(0, post_event_daily_noise, post_gap_n + 1)
        )

        # Pin noise to zero at both ends so the final level stays controlled.
        trend_noise = trend_noise - np.linspace(
            trend_noise[0],
            trend_noise[-1],
            post_gap_n + 1
        )

        price[gap_idx:] = trend_line + trend_noise

    return price


# ============================================================
# Generate counterfactual path: sideways, then down
# ============================================================

def build_sideways_down_counterfactual_path(
    rng,
    n_days,
    initial_price=100,
    transition_idx=112,
    final_price=91,
    pre_noise=0.06,
    post_noise=0.08
):
    """
    Builds the counterfactual path:

    1. Horizontal / sideways market.
    2. No gap up.
    3. Sideways-to-down drift over the same timeline.
    """
    price = np.empty(n_days)
    price[0] = initial_price

    # Sideways before the transition point
    for i in range(1, transition_idx):
        mean_reversion = 0.050 * (initial_price - price[i - 1])
        noise = rng.normal(0, pre_noise)
        price[i] = price[i - 1] + mean_reversion + noise

    # Sideways/down after the same event window
    post_n = n_days - transition_idx

    start_price = price[transition_idx - 1]
    down_line = np.linspace(start_price, final_price, post_n)

    down_noise = np.cumsum(
        rng.normal(0, post_noise, post_n)
    )

    # Pin noise to zero at both ends so the final level remains controlled.
    down_noise = down_noise - np.linspace(
        down_noise[0],
        down_noise[-1],
        post_n
    )

    price[transition_idx:] = down_line + down_noise

    return price


observed_price = build_gap_up_trend_path(
    rng=rng,
    n_days=n_days,
    initial_price=INITIAL_PRICE,
    gap_idx=gap_idx,
    gap_up_pct=GAP_UP_PCT,
    post_gap_total_trend_pct=POST_GAP_TOTAL_TREND_PCT,
    pre_event_daily_noise=PRE_EVENT_DAILY_NOISE,
    post_event_daily_noise=POST_EVENT_DAILY_NOISE
)

counterfactual_price = build_sideways_down_counterfactual_path(
    rng=rng,
    n_days=n_days,
    initial_price=INITIAL_PRICE,
    transition_idx=gap_idx,
    final_price=COUNTERFACTUAL_FINAL_PRICE,
    pre_noise=COUNTERFACTUAL_PRE_NOISE,
    post_noise=COUNTERFACTUAL_POST_NOISE
)

# ============================================================
# Build DataFrame
# ============================================================

df = pd.DataFrame({
    "date": dates,
    "observed_price": observed_price,
    "counterfactual_price": counterfactual_price
})

df["observed_return"] = df["observed_price"] / INITIAL_PRICE - 1
df["counterfactual_return"] = df["counterfactual_price"] / INITIAL_PRICE - 1

# ============================================================
# Summary stats
# ============================================================

latest_date = df["date"].iloc[-1]

observed_final_return = df["observed_return"].iloc[-1]
counterfactual_final_return = df["counterfactual_return"].iloc[-1]

observed_gap_return = (
    df["observed_price"].iloc[gap_idx]
    / df["observed_price"].iloc[gap_idx - 1]
    - 1
)

observed_final_price = df["observed_price"].iloc[-1]
counterfactual_final_price = df["counterfactual_price"].iloc[-1]

def max_drawdown_from_price(price):
    price = np.asarray(price)
    running_max = np.maximum.accumulate(price)
    drawdown = 1 - price / running_max
    return np.max(drawdown)

observed_mdd = max_drawdown_from_price(df["observed_price"])
counterfactual_mdd = max_drawdown_from_price(df["counterfactual_price"])

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

observed_color = "#ffaa33"
counterfactual_color = "#00ff88"

baseline_color = "#777777"
indicator_color_red = "#ff4444"
indicator_color_green = "#34ff34"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(
        f"{LEFT_PATH_NAME}: Final {observed_final_return:.1%} | "
        f"Gap {observed_gap_return:.1%} | "
        f"MDD {observed_mdd:.1%}",

        f"{RIGHT_PATH_NAME}: Final {counterfactual_final_return:.1%} | "
        f"No Gap | "
        f"MDD {counterfactual_mdd:.1%}"
    )
)

# ============================================================
# Static baseline and indicator markers
# ============================================================

# Baseline at initial price
fig.add_hline(
    y=INITIAL_PRICE,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1
)

fig.add_hline(
    y=INITIAL_PRICE,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2
)

# Buy Indicator vertical line on left (observed) chart (GREEN)
fig.add_shape(
    type="line",
    x0=buy_indicator_x,
    x1=buy_indicator_x,
    y0=LEFT_Y_RANGE[0],
    y1=LEFT_Y_RANGE[1],
    line=dict(color=indicator_color_green, width=2, dash="dot"),
    opacity=0.95,
    row=1,
    col=1
)

# Buy Indicator vertical line on right (counterfactual) chart (RED)
fig.add_shape(
    type="line",
    x0=buy_indicator_x,
    x1=buy_indicator_x,
    y0=RIGHT_Y_RANGE[0],
    y1=RIGHT_Y_RANGE[1],
    line=dict(color=indicator_color_red, width=2, dash="dot"),
    opacity=0.95,
    row=1,
    col=2
)

# --- Slide down the arrow and label to the middle of the chart ---

# For left (observed), center y position for arrow and label
mid_left_y = (LEFT_Y_RANGE[0] + LEFT_Y_RANGE[1]) / 2

fig.add_annotation(
    x=buy_indicator_x,
    y=mid_left_y,
    ax=-75,
    ay=0,  # Arrow is now horizontal, pointing from left
    text=BUY_INDICATOR_LABEL_LEFT,
    showarrow=True,
    arrowhead=3,
    arrowsize=1.2,
    arrowwidth=2,
    arrowcolor=indicator_color_green,
    font=dict(color=indicator_color_green, size=13),
    bgcolor="rgba(0,0,0,0.35)",
    bordercolor=indicator_color_green,
    borderwidth=1,
    row=1,
    col=1
)

# For right (counterfactual), center y position for arrow and label
mid_right_y = (RIGHT_Y_RANGE[0] + RIGHT_Y_RANGE[1]) / 2

fig.add_annotation(
    x=buy_indicator_x,
    y=mid_right_y,
    ax=75,
    ay=0,  # Arrow is now horizontal, pointing from right
    text=BUY_INDICATOR_LABEL_RIGHT,
    showarrow=True,
    arrowhead=3,
    arrowsize=1.2,
    arrowwidth=2,
    arrowcolor=indicator_color_red,
    font=dict(color=indicator_color_red, size=13),
    bgcolor="rgba(0,0,0,0.35)",
    bordercolor=indicator_color_red,
    borderwidth=1,
    row=1,
    col=2
)

# ============================================================
# Initial traces
# ============================================================

# Left: observed market path
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["observed_price"].iloc[0]],
        mode="lines",
        line=dict(color=observed_color, width=4),
        name=LEFT_PATH_NAME,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{LEFT_PATH_NAME}: $%{{y:.2f}}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# Right: counterfactual sideways/down path
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["counterfactual_price"].iloc[0]],
        mode="lines",
        line=dict(color=counterfactual_color, width=4),
        name=RIGHT_PATH_NAME,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{RIGHT_PATH_NAME}: $%{{y:.2f}}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(1, len(df), FRAME_STRIDE))

if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    plot_slice = df.iloc[:i + 1]

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=plot_slice["date"],
                    y=plot_slice["observed_price"]
                ),
                go.Scatter(
                    x=plot_slice["date"],
                    y=plot_slice["counterfactual_price"]
                )
            ],
            traces=[0, 1],
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["date"].iloc[i].strftime("%b %Y"),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "Buy Indicator vs. Sideways/Down Counterfactual"
            "<br><sup>"
            f"{START_DATE} to {latest_date:%Y-%m-%d} · "
            f"Buy Indicator: {buy_indicator_date:%Y-%m-%d} · "
            f"Observed Final: ${observed_final_price:.2f} · "
            f"Counterfactual Final: ${counterfactual_final_price:.2f}"
            "</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=FIG_HEIGHT,
    width=FIG_WIDTH,
    margin=dict(t=120, b=150, r=50, l=75),
    showlegend=False,
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": False
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {
                            "duration": 0,
                            "redraw": False
                        },
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_annotations(font=dict(size=14))

# ============================================================
# Axes
# ============================================================

# Left subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=LEFT_Y_RANGE,
    title_text="Price Index, Starting at $100",
    tickprefix="$",
    tickformat=",.0f"
)

# Right subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=RIGHT_Y_RANGE,
    title_text="Counterfactual Price Index",
    tickprefix="$",
    tickformat=",.0f"
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("buy_indicator_vs_sideways_down_counterfactual_animated.html", include_plotlyjs="cdn")

###### ______________________________________________________________________________________________________________________________________

##### Without a Counterfactual Probabilities and Statistics are Messy

In reality everything is uncertain without a crystal ball, we don't have nice asymptotic convergence like in the classroom.

Practical implications of this: backtest distributions have no reason to converge going forward (even conditioning on regimes).

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

SEED = 42

LEFT_PATH_NAME = "In-Sample Buy Indicator"
RIGHT_PATH_NAME = "Out-of-Sample Buy Indicator"

NUM_TRADES = 520

# In-sample signal behavior: positive expectation
INSAMPLE_WIN_PROBABILITY = 0.58
INSAMPLE_AVG_WIN_PCT = 1.25
INSAMPLE_AVG_LOSS_PCT = 0.95

# Out-of-sample signal behavior: negative expectation
OOS_WIN_PROBABILITY = 0.44
OOS_AVG_WIN_PCT = 1.10
OOS_AVG_LOSS_PCT = 1.20

# Distribution shape settings
WIN_LOGNORMAL_SIGMA = 0.42
LOSS_LOGNORMAL_SIGMA = 0.48
TAIL_SHOCK_PROBABILITY = 0.035
TAIL_SHOCK_SCALE = 2.25

# Histogram settings
BIN_MIN = -6.0
BIN_MAX = 6.0
NUM_BINS = 36

# Animation settings
INITIAL_FRAME_SIZE = 20
FRAME_STRIDE = 10
FRAME_DURATION = 35

# Figure settings
FIG_WIDTH = 1200
FIG_HEIGHT = 650

# ============================================================
# Randomness
# ============================================================

rng = np.random.default_rng(SEED)

# ============================================================
# Helper functions
# ============================================================

def expected_value_pct(
    win_probability,
    avg_win_pct,
    avg_loss_pct
):
    return (
        win_probability * avg_win_pct
        - (1 - win_probability) * avg_loss_pct
    )


def generate_buy_indicator_pl_distribution(
    rng,
    n_trades,
    win_probability,
    avg_win_pct,
    avg_loss_pct,
    target_ev_pct,
    win_sigma=0.42,
    loss_sigma=0.48,
    tail_shock_probability=0.035,
    tail_shock_scale=2.25
):
    """
    Generates a synthetic per-trade P/L distribution in percentage points.

    The distribution is intentionally skewed and noisy:
    - wins are positive and variable
    - losses are negative and variable
    - occasional tail events appear on both sides

    The final distribution is shifted slightly so the realized sample mean
    equals the intended expectancy.
    """
    wins = rng.random(n_trades) < win_probability

    win_mu = np.log(avg_win_pct) - 0.5 * win_sigma ** 2
    loss_mu = np.log(avg_loss_pct) - 0.5 * loss_sigma ** 2

    win_pl = rng.lognormal(
        mean=win_mu,
        sigma=win_sigma,
        size=n_trades
    )

    loss_pl = -rng.lognormal(
        mean=loss_mu,
        sigma=loss_sigma,
        size=n_trades
    )

    pl = np.where(wins, win_pl, loss_pl)

    # Add occasional tails to make the P/L distribution look more realistic.
    tail_events = rng.random(n_trades) < tail_shock_probability
    tail_signs = rng.choice([-1, 1], size=n_trades)
    tail_sizes = rng.exponential(tail_shock_scale, size=n_trades)

    pl = pl + tail_events * tail_signs * tail_sizes

    # Clip extreme tails so the histogram stays legible.
    pl = np.clip(pl, BIN_MIN + 0.05, BIN_MAX - 0.05)

    # Force realized mean to match intended expectation.
    pl = pl + (target_ev_pct - np.mean(pl))

    # Clip one more time after mean adjustment.
    pl = np.clip(pl, BIN_MIN + 0.05, BIN_MAX - 0.05)

    return pl


def histogram_counts(values, bins):
    counts, _ = np.histogram(values, bins=bins)
    return counts


def win_rate(values):
    return np.mean(np.asarray(values) > 0)


# ============================================================
# Generate distributions
# ============================================================

insample_target_ev = expected_value_pct(
    INSAMPLE_WIN_PROBABILITY,
    INSAMPLE_AVG_WIN_PCT,
    INSAMPLE_AVG_LOSS_PCT
)

oos_target_ev = expected_value_pct(
    OOS_WIN_PROBABILITY,
    OOS_AVG_WIN_PCT,
    OOS_AVG_LOSS_PCT
)

insample_pl = generate_buy_indicator_pl_distribution(
    rng=rng,
    n_trades=NUM_TRADES,
    win_probability=INSAMPLE_WIN_PROBABILITY,
    avg_win_pct=INSAMPLE_AVG_WIN_PCT,
    avg_loss_pct=INSAMPLE_AVG_LOSS_PCT,
    target_ev_pct=insample_target_ev,
    win_sigma=WIN_LOGNORMAL_SIGMA,
    loss_sigma=LOSS_LOGNORMAL_SIGMA,
    tail_shock_probability=TAIL_SHOCK_PROBABILITY,
    tail_shock_scale=TAIL_SHOCK_SCALE
)

oos_pl = generate_buy_indicator_pl_distribution(
    rng=rng,
    n_trades=NUM_TRADES,
    win_probability=OOS_WIN_PROBABILITY,
    avg_win_pct=OOS_AVG_WIN_PCT,
    avg_loss_pct=OOS_AVG_LOSS_PCT,
    target_ev_pct=oos_target_ev,
    win_sigma=WIN_LOGNORMAL_SIGMA,
    loss_sigma=LOSS_LOGNORMAL_SIGMA,
    tail_shock_probability=TAIL_SHOCK_PROBABILITY,
    tail_shock_scale=TAIL_SHOCK_SCALE
)

insample_realized_mean = np.mean(insample_pl)
oos_realized_mean = np.mean(oos_pl)

insample_realized_win_rate = win_rate(insample_pl)
oos_realized_win_rate = win_rate(oos_pl)

insample_median = np.median(insample_pl)
oos_median = np.median(oos_pl)

# ============================================================
# Histogram setup
# ============================================================

bins = np.linspace(BIN_MIN, BIN_MAX, NUM_BINS + 1)
bin_centers = (bins[:-1] + bins[1:]) / 2
bin_width = bins[1] - bins[0]

insample_full_counts = histogram_counts(insample_pl, bins)
oos_full_counts = histogram_counts(oos_pl, bins)

y_axis_max = max(
    insample_full_counts.max(),
    oos_full_counts.max()
)

y_axis_upper = int(np.ceil(y_axis_max * 1.20))

# Initial animation slice
initial_n = min(INITIAL_FRAME_SIZE, NUM_TRADES)

initial_insample_counts = histogram_counts(insample_pl[:initial_n], bins)
initial_oos_counts = histogram_counts(oos_pl[:initial_n], bins)

initial_insample_mean = np.mean(insample_pl[:initial_n])
initial_oos_mean = np.mean(oos_pl[:initial_n])

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

insample_bar_color = "#00ff88"
oos_bar_color = "#ff4444"

insample_mean_color = "#aaffcc"
oos_mean_color = "#ffaaaa"

zero_line_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(
        (
            f"{LEFT_PATH_NAME}: "
            f"Mean {insample_realized_mean:+.2f}% | "
            f"Win Rate {insample_realized_win_rate:.1%}"
        ),
        (
            f"{RIGHT_PATH_NAME}: "
            f"Mean {oos_realized_mean:+.2f}% | "
            f"Win Rate {oos_realized_win_rate:.1%}"
        )
    )
)

# ============================================================
# Static zero P/L reference lines
# ============================================================

for col in [1, 2]:
    fig.add_shape(
        type="line",
        x0=0,
        x1=0,
        y0=0,
        y1=y_axis_upper,
        line=dict(color=zero_line_color, width=1.5, dash="dash"),
        opacity=0.75,
        row=1,
        col=col
    )

# ============================================================
# Initial traces
# ============================================================

# Left: in-sample histogram
fig.add_trace(
    go.Bar(
        x=bin_centers,
        y=initial_insample_counts,
        width=bin_width * 0.92,
        marker=dict(
            color=insample_bar_color,
            line=dict(color="rgba(255,255,255,0.15)", width=0.5)
        ),
        name="In-Sample P/L",
        hovertemplate=(
            "P/L Bin Center: %{x:.2f}%<br>"
            "Trade Count: %{y}<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=1
)

# Left: in-sample mean line
fig.add_trace(
    go.Scatter(
        x=[initial_insample_mean, initial_insample_mean],
        y=[0, y_axis_upper],
        mode="lines",
        line=dict(color=insample_mean_color, width=3),
        name="In-Sample Mean",
        hovertemplate=(
            "In-Sample Mean: %{x:.2f}%<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=1
)

# Right: out-of-sample histogram
fig.add_trace(
    go.Bar(
        x=bin_centers,
        y=initial_oos_counts,
        width=bin_width * 0.92,
        marker=dict(
            color=oos_bar_color,
            line=dict(color="rgba(255,255,255,0.15)", width=0.5)
        ),
        name="Out-of-Sample P/L",
        hovertemplate=(
            "P/L Bin Center: %{x:.2f}%<br>"
            "Trade Count: %{y}<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=2
)

# Right: out-of-sample mean line
fig.add_trace(
    go.Scatter(
        x=[initial_oos_mean, initial_oos_mean],
        y=[0, y_axis_upper],
        mode="lines",
        line=dict(color=oos_mean_color, width=3),
        name="Out-of-Sample Mean",
        hovertemplate=(
            "Out-of-Sample Mean: %{x:.2f}%<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=2
)

# Trace indices
INSAMPLE_HIST_TRACE_INDEX = 0
INSAMPLE_MEAN_TRACE_INDEX = 1
OOS_HIST_TRACE_INDEX = 2
OOS_MEAN_TRACE_INDEX = 3

# ============================================================
# In-chart legend labels
# ============================================================

legend_y_top = y_axis_upper * 0.96
legend_y_second = y_axis_upper * 0.88
legend_x_left = BIN_MIN + 0.35

# Left in-chart legend
fig.add_annotation(
    x=legend_x_left,
    y=legend_y_top,
    text="■ In-Sample P/L",
    showarrow=False,
    font=dict(color=insample_bar_color, size=13),
    bgcolor="rgba(0,0,0,0.35)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="middle",
    row=1,
    col=1
)

fig.add_annotation(
    x=legend_x_left,
    y=legend_y_second,
    text="━ Mean",
    showarrow=False,
    font=dict(color=insample_mean_color, size=13),
    bgcolor="rgba(0,0,0,0.35)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="middle",
    row=1,
    col=1
)

# Right in-chart legend
fig.add_annotation(
    x=legend_x_left,
    y=legend_y_top,
    text="■ Out-of-Sample P/L",
    showarrow=False,
    font=dict(color=oos_bar_color, size=13),
    bgcolor="rgba(0,0,0,0.35)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="middle",
    row=1,
    col=2
)

fig.add_annotation(
    x=legend_x_left,
    y=legend_y_second,
    text="━ Mean",
    showarrow=False,
    font=dict(color=oos_mean_color, size=13),
    bgcolor="rgba(0,0,0,0.35)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="middle",
    row=1,
    col=2
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(INITIAL_FRAME_SIZE, NUM_TRADES + 1, FRAME_STRIDE))

if not frame_indices or frame_indices[-1] != NUM_TRADES:
    frame_indices.append(NUM_TRADES)

for n in frame_indices:
    frame_name = f"n{n}"

    insample_counts = histogram_counts(insample_pl[:n], bins)
    oos_counts = histogram_counts(oos_pl[:n], bins)

    insample_mean = np.mean(insample_pl[:n])
    oos_mean = np.mean(oos_pl[:n])

    frames.append(
        go.Frame(
            data=[
                go.Bar(
                    x=bin_centers,
                    y=insample_counts
                ),
                go.Scatter(
                    x=[insample_mean, insample_mean],
                    y=[0, y_axis_upper]
                ),
                go.Bar(
                    x=bin_centers,
                    y=oos_counts
                ),
                go.Scatter(
                    x=[oos_mean, oos_mean],
                    y=[0, y_axis_upper]
                )
            ],
            traces=[
                INSAMPLE_HIST_TRACE_INDEX,
                INSAMPLE_MEAN_TRACE_INDEX,
                OOS_HIST_TRACE_INDEX,
                OOS_MEAN_TRACE_INDEX
            ],
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": f"{n}",
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "Buy Indicator P/L Distribution: In-Sample Edge vs. Out-of-Sample Decay"
            "<br><sup>"
            f"In-Sample: EV {insample_realized_mean:+.2f}% / trade, "
            f"Median {insample_median:+.2f}%, "
            f"Win Rate {insample_realized_win_rate:.1%} · "
            f"Out-of-Sample: EV {oos_realized_mean:+.2f}% / trade, "
            f"Median {oos_median:+.2f}%, "
            f"Win Rate {oos_realized_win_rate:.1%}"
            "</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=FIG_HEIGHT,
    width=FIG_WIDTH,
    margin=dict(t=120, b=115, r=50, l=75),
    bargap=0.03,
    showlegend=False,
    hovermode="closest",

    # Buttons closer to the bottom of the charts.
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": False
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {
                            "duration": 0,
                            "redraw": False
                        },
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 46},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": -0.08,
        "yanchor": "top"
    }],

    # Slider closer to the bottom of the charts.
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Trades Shown: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 18},
        "len": 0.85,
        "x": 0.15,
        "y": -0.08,
        "steps": slider_steps
    }]
)

fig.update_annotations(font=dict(size=14))

# ============================================================
# Axes
# ============================================================

# Left subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[BIN_MIN, BIN_MAX],
    title_text="Per-Trade P/L After Buy Indicator"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[0, y_axis_upper],
    title_text="Trade Count"
)

# Right subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[BIN_MIN, BIN_MAX],
    title_text="Per-Trade P/L After Buy Indicator"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[0, y_axis_upper],
    title_text="Trade Count"
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("buy_indicator_insample_vs_oos_pl_distribution.html", include_plotlyjs="cdn")

###### ______________________________________________________________________________________________________________________________________

##### It's About Stability of Distributions, Probabilities, Statistics, and Position Sizing

$$
\mathbb{E}[\text{Trade P/L}] = \mathbb{E}[\text{P/L} \mid \text{Win}] \cdot \mathbb{P}(\text{Win}) + \mathbb{E}[\text{P/L} \mid \text{Loss}] \cdot \mathbb{P}(\text{Loss})
$$

In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

# ============================================================
# Config
# ============================================================

START_DATE = "2025-01-01"
END_DATE = "2026-12-31"

SEED = 42
INITIAL_WEALTH = 100

LEFT_PATH_NAME = "Strategy Expected Value"
RIGHT_PATH_NAME = "Trading Wealth Path"

# Default strategy assumptions
DEFAULT_WIN_PROBABILITY = 55      # percent
DEFAULT_WIN_SIZE = 2.00           # percent gain when right
DEFAULT_LOSS_SIZE = 1.50          # percent loss when wrong

# Slider ranges
MIN_WIN_PROBABILITY = 30
MAX_WIN_PROBABILITY = 75
STEP_WIN_PROBABILITY = 1

MIN_WIN_SIZE = 0.25
MAX_WIN_SIZE = 5.00
STEP_WIN_SIZE = 0.25

MIN_LOSS_SIZE = 0.25
MAX_LOSS_SIZE = 5.00
STEP_LOSS_SIZE = 0.25

# Animation settings
FRAME_DURATION_MS = 18
FRAME_STRIDE = 4

# Figure settings
FIG_WIDTH = 1200
FIG_HEIGHT = 650

# ============================================================
# Generate synthetic trading dates and fixed randomness
# ============================================================

dates = pd.bdate_range(START_DATE, END_DATE)

if len(dates) < 2:
    raise ValueError("Need at least two business days to create an animated wealth path.")

rng = np.random.default_rng(SEED)
n_days = len(dates)

# Fixed random stream:
# The sliders change the strategy parameters, but the underlying sequence
# of wins/losses stays constant for apples-to-apples comparison.
random_uniforms = rng.random(n_days)
random_uniforms[0] = 1.0

date_strings = [d.strftime("%Y-%m-%d") for d in dates]

# ============================================================
# Helper functions
# ============================================================

def simulate_wealth_path(
    random_uniforms,
    win_probability_pct,
    win_size_pct,
    loss_size_pct,
    initial_wealth=100
):
    p_win = win_probability_pct / 100
    win_return = win_size_pct / 100
    loss_return = loss_size_pct / 100

    wins = random_uniforms < p_win
    trade_returns = np.where(wins, win_return, -loss_return)
    trade_returns[0] = 0

    wealth = initial_wealth * np.cumprod(1 + trade_returns)

    return wealth, trade_returns, wins


def strategy_expected_values(
    win_probability_pct,
    win_size_pct,
    loss_size_pct
):
    p_win = win_probability_pct / 100
    p_loss = 1 - p_win

    win_contribution = p_win * win_size_pct
    loss_contribution = -p_loss * loss_size_pct
    net_ev = win_contribution + loss_contribution

    return win_contribution, loss_contribution, net_ev


def max_drawdown_from_wealth(wealth):
    wealth = np.asarray(wealth)
    running_max = np.maximum.accumulate(wealth)
    drawdown = 1 - wealth / running_max
    return np.max(drawdown)


def annualized_sharpe(simple_returns):
    simple_returns = np.asarray(simple_returns)

    if len(simple_returns) > 1 and simple_returns[0] == 0:
        simple_returns = simple_returns[1:]

    if len(simple_returns) < 2:
        return np.nan

    mean = np.mean(simple_returns)
    std = np.std(simple_returns, ddof=1)

    if std == 0:
        return np.nan

    return mean / std * np.sqrt(252)


def annualized_cagr(final_wealth, initial_wealth, n_days):
    years = n_days / 252
    return (final_wealth / initial_wealth) ** (1 / years) - 1


# ============================================================
# Initial strategy calculation
# ============================================================

initial_wealth_path, initial_returns, initial_wins = simulate_wealth_path(
    random_uniforms=random_uniforms,
    win_probability_pct=DEFAULT_WIN_PROBABILITY,
    win_size_pct=DEFAULT_WIN_SIZE,
    loss_size_pct=DEFAULT_LOSS_SIZE,
    initial_wealth=INITIAL_WEALTH
)

initial_win_contribution, initial_loss_contribution, initial_net_ev = strategy_expected_values(
    DEFAULT_WIN_PROBABILITY,
    DEFAULT_WIN_SIZE,
    DEFAULT_LOSS_SIZE
)

initial_final_wealth = initial_wealth_path[-1]
initial_mdd = max_drawdown_from_wealth(initial_wealth_path)
initial_sharpe = annualized_sharpe(initial_returns)

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

bar_win_color = "#00ff88"
bar_loss_color = "#ff4444"
bar_ev_color = "#ffaa33"

wealth_color = "#00ff88"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

bar_x = [
    "Expected Win",
    "Expected Loss",
    "Net EV"
]

bar_y = [
    initial_win_contribution,
    initial_loss_contribution,
    initial_net_ev
]

bar_colors = [
    bar_win_color,
    bar_loss_color,
    bar_ev_color
]

bar_min = min(min(bar_y), 0)
bar_max = max(max(bar_y), 0)
bar_pad = max((bar_max - bar_min) * 0.25, 0.25)

wealth_y_pad = max(
    (initial_wealth_path.max() - initial_wealth_path.min()) * 0.15,
    10
)
wealth_y_lower = max(0, initial_wealth_path.min() - wealth_y_pad)
wealth_y_upper = initial_wealth_path.max() + wealth_y_pad

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.42, 0.58],
    horizontal_spacing=0.10,
    subplot_titles=(
        (
            f"{LEFT_PATH_NAME}: "
            f"EV {initial_net_ev:.2f}% / trade"
        ),
        (
            f"{RIGHT_PATH_NAME}: "
            "Press Play to animate"
        )
    )
)

# Left: EV bar chart
fig.add_trace(
    go.Bar(
        x=bar_x,
        y=bar_y,
        marker=dict(color=bar_colors),
        text=[f"{v:.2f}%" for v in bar_y],
        textposition="outside",
        cliponaxis=False,
        hovertemplate="%{x}<br>%{y:.2f}% per trade<extra></extra>",
        name=LEFT_PATH_NAME
    ),
    row=1,
    col=1
)

# Right: wealth path starts with only first point.
# The Play button reveals the rest.
fig.add_trace(
    go.Scatter(
        x=[dates[0]],
        y=[initial_wealth_path[0]],
        mode="lines",
        line=dict(color=wealth_color, width=4),
        name=RIGHT_PATH_NAME,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Wealth: $%{y:.2f}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# Baselines
fig.add_hline(
    y=0,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1
)

fig.add_hline(
    y=INITIAL_WEALTH,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2
)

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "Expected Value Inputs vs. Simulated Trading Wealth Path"
            "<br><sup>"
            f"Initial Wealth: ${INITIAL_WEALTH:.0f} · "
            f"Default P(win): {DEFAULT_WIN_PROBABILITY:.0f}% · "
            f"Default Win: {DEFAULT_WIN_SIZE:.2f}% · "
            f"Default Loss: {DEFAULT_LOSS_SIZE:.2f}% · "
            f"Default EV: {initial_net_ev:.2f}% / trade"
            "</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=FIG_HEIGHT,
    width=FIG_WIDTH,
    margin=dict(t=120, b=105, r=50, l=75),
    showlegend=False,
    hovermode="closest"
)

fig.update_annotations(font=dict(color=off_white, size=14))

# Left subplot axes
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    title_text=""
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=[bar_min - bar_pad, bar_max + bar_pad],
    title_text="Expected Return Contribution Per Trade",
    ticksuffix="%",
    tickformat=".2f"
)

# Right subplot axes
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[dates[0], dates[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=[wealth_y_lower, wealth_y_upper],
    title_text="Wealth, Starting at $100",
    tickprefix="$",
    tickformat=",.0f"
)

# ============================================================
# HTML + JavaScript controls
# ============================================================

div_id = "ev_strategy_slider_animation_fixed"

plot_html = fig.to_html(
    full_html=False,
    include_plotlyjs="cdn",
    div_id=div_id,
    config={
        "displayModeBar": False,
        "responsive": True
    }
)

payload = {
    "dates": date_strings,
    "random_uniforms": random_uniforms.tolist(),
    "initial_wealth": INITIAL_WEALTH,
    "frame_duration_ms": FRAME_DURATION_MS,
    "frame_stride": FRAME_STRIDE,
    "colors": {
        "win": bar_win_color,
        "loss": bar_loss_color,
        "ev": bar_ev_color,
        "wealth": wealth_color,
        "baseline": baseline_color,
        "off_white": off_white
    }
}

payload_json = json.dumps(payload)

controls_html = f"""
<style>
    .ev-control-panel {{
        width: {FIG_WIDTH}px;
        max-width: 100%;
        margin: 0 auto 14px auto;
        padding: 14px 18px;
        border: 1px solid rgba(255,255,255,0.14);
        border-radius: 12px;
        background: rgba(0,0,0,0.28);
        color: {off_white};
        font-family: Arial, sans-serif;
        box-sizing: border-box;
    }}
    .ev-control-grid {{
        display: grid;
        grid-template-columns: 1fr 1fr 1fr;
        gap: 18px;
        align-items: center;
    }}
    .ev-control {{
        display: flex;
        flex-direction: column;
        gap: 6px;
    }}
    .ev-control label {{
        font-size: 13px;
        color: {off_white};
        display: flex;
        justify-content: space-between;
        gap: 8px;
    }}
    .ev-control input[type="range"] {{
        width: 100%;
    }}
    .ev-metrics {{
        display: grid;
        grid-template-columns: repeat(5, 1fr);
        gap: 10px;
        margin-top: 14px;
        font-size: 13px;
    }}
    .ev-metric {{
        padding: 8px 10px;
        border-radius: 8px;
        background: rgba(255,255,255,0.06);
        border: 1px solid rgba(255,255,255,0.08);
    }}
    .ev-metric .label {{
        color: rgba(224,224,224,0.75);
        font-size: 11px;
        margin-bottom: 3px;
    }}
    .ev-metric .value {{
        color: {off_white};
        font-size: 14px;
        font-weight: 600;
    }}
    .ev-buttons {{
        display: flex;
        gap: 10px;
        margin-top: 14px;
    }}
    .ev-buttons button {{
        cursor: pointer;
        border: 1px solid rgba(255,255,255,0.20);
        border-radius: 8px;
        padding: 8px 12px;
        color: {off_white};
        background: rgba(255,255,255,0.08);
        font-size: 13px;
    }}
    .ev-buttons button:hover {{
        background: rgba(255,255,255,0.14);
    }}
</style>

<div class="ev-control-panel">
    <div class="ev-control-grid">
        <div class="ev-control">
            <label>
                <span>P(win)</span>
                <strong><span id="winProbValue">{DEFAULT_WIN_PROBABILITY}</span>%</strong>
            </label>
            <input
                id="winProbSlider"
                type="range"
                min="{MIN_WIN_PROBABILITY}"
                max="{MAX_WIN_PROBABILITY}"
                step="{STEP_WIN_PROBABILITY}"
                value="{DEFAULT_WIN_PROBABILITY}"
            >
        </div>

        <div class="ev-control">
            <label>
                <span>Win size</span>
                <strong><span id="winSizeValue">{DEFAULT_WIN_SIZE:.2f}</span>%</strong>
            </label>
            <input
                id="winSizeSlider"
                type="range"
                min="{MIN_WIN_SIZE}"
                max="{MAX_WIN_SIZE}"
                step="{STEP_WIN_SIZE}"
                value="{DEFAULT_WIN_SIZE}"
            >
        </div>

        <div class="ev-control">
            <label>
                <span>Loss size</span>
                <strong><span id="lossSizeValue">{DEFAULT_LOSS_SIZE:.2f}</span>%</strong>
            </label>
            <input
                id="lossSizeSlider"
                type="range"
                min="{MIN_LOSS_SIZE}"
                max="{MAX_LOSS_SIZE}"
                step="{STEP_LOSS_SIZE}"
                value="{DEFAULT_LOSS_SIZE}"
            >
        </div>
    </div>

    <div class="ev-metrics">
        <div class="ev-metric">
            <div class="label">Expected value</div>
            <div class="value" id="evMetric">--</div>
        </div>
        <div class="ev-metric">
            <div class="label">Expected R multiple</div>
            <div class="value" id="rMetric">--</div>
        </div>
        <div class="ev-metric">
            <div class="label">Final wealth</div>
            <div class="value" id="wealthMetric">--</div>
        </div>
        <div class="ev-metric">
            <div class="label">Realized win rate</div>
            <div class="value" id="winRateMetric">--</div>
        </div>
        <div class="ev-metric">
            <div class="label">Max drawdown</div>
            <div class="value" id="mddMetric">--</div>
        </div>
    </div>

    <div class="ev-buttons">
        <button id="animateStrategyButton">▶ Play Animation</button>
        <button id="showFullPathButton">Show Full Path</button>
        <button id="resetStrategyButton">↺ Reset Defaults</button>
    </div>
</div>
"""

script_template = r"""
<script>
(function() {
    const payload = __PAYLOAD_JSON__;
    const plotDiv = document.getElementById("__DIV_ID__");

    const winProbSlider = document.getElementById("winProbSlider");
    const winSizeSlider = document.getElementById("winSizeSlider");
    const lossSizeSlider = document.getElementById("lossSizeSlider");

    const winProbValue = document.getElementById("winProbValue");
    const winSizeValue = document.getElementById("winSizeValue");
    const lossSizeValue = document.getElementById("lossSizeValue");

    const evMetric = document.getElementById("evMetric");
    const rMetric = document.getElementById("rMetric");
    const wealthMetric = document.getElementById("wealthMetric");
    const winRateMetric = document.getElementById("winRateMetric");
    const mddMetric = document.getElementById("mddMetric");

    const animateStrategyButton = document.getElementById("animateStrategyButton");
    const showFullPathButton = document.getElementById("showFullPathButton");
    const resetStrategyButton = document.getElementById("resetStrategyButton");

    let animationTimer = null;

    function formatPct(x, digits = 2) {
        return x.toFixed(digits) + "%";
    }

    function formatDollar(x) {
        return "$" + x.toFixed(2);
    }

    function maxDrawdown(wealth) {
        let runningMax = wealth[0];
        let maxDd = 0;

        for (let i = 0; i < wealth.length; i++) {
            runningMax = Math.max(runningMax, wealth[i]);
            const dd = 1 - wealth[i] / runningMax;
            maxDd = Math.max(maxDd, dd);
        }

        return maxDd;
    }

    function simulate() {
        const pWinPct = parseFloat(winProbSlider.value);
        const winSizePct = parseFloat(winSizeSlider.value);
        const lossSizePct = parseFloat(lossSizeSlider.value);

        const pWin = pWinPct / 100;
        const pLoss = 1 - pWin;

        const winReturn = winSizePct / 100;
        const lossReturn = lossSizePct / 100;

        const wealth = new Array(payload.random_uniforms.length);
        const returns = new Array(payload.random_uniforms.length);
        const wins = new Array(payload.random_uniforms.length);

        wealth[0] = payload.initial_wealth;
        returns[0] = 0;
        wins[0] = false;

        for (let i = 1; i < payload.random_uniforms.length; i++) {
            const isWin = payload.random_uniforms[i] < pWin;
            const r = isWin ? winReturn : -lossReturn;

            wins[i] = isWin;
            returns[i] = r;
            wealth[i] = wealth[i - 1] * (1 + r);
        }

        const winContribution = pWin * winSizePct;
        const lossContribution = -pLoss * lossSizePct;
        const netEv = winContribution + lossContribution;

        const realizedWinRate = wins.slice(1).filter(Boolean).length / (wins.length - 1);
        const finalWealth = wealth[wealth.length - 1];
        const mdd = maxDrawdown(wealth);
        const rMultiple = lossSizePct === 0 ? 0 : netEv / lossSizePct;

        return {
            pWinPct,
            winSizePct,
            lossSizePct,
            winContribution,
            lossContribution,
            netEv,
            rMultiple,
            wealth,
            returns,
            wins,
            realizedWinRate,
            finalWealth,
            mdd
        };
    }

    function wealthYRange(wealth) {
        const minVal = Math.min(...wealth);
        const maxVal = Math.max(...wealth);
        const pad = Math.max((maxVal - minVal) * 0.15, 10);

        return [
            Math.max(0, minVal - pad),
            maxVal + pad
        ];
    }

    function barYRange(values) {
        const minVal = Math.min(...values, 0);
        const maxVal = Math.max(...values, 0);
        const pad = Math.max((maxVal - minVal) * 0.25, 0.25);

        return [
            minVal - pad,
            maxVal + pad
        ];
    }

    function updateControlsText(result) {
        winProbValue.textContent = result.pWinPct.toFixed(0);
        winSizeValue.textContent = result.winSizePct.toFixed(2);
        lossSizeValue.textContent = result.lossSizePct.toFixed(2);

        evMetric.textContent = formatPct(result.netEv);
        rMetric.textContent = result.rMultiple.toFixed(2) + "R";
        wealthMetric.textContent = formatDollar(result.finalWealth);
        winRateMetric.textContent = formatPct(result.realizedWinRate * 100, 1);
        mddMetric.textContent = formatPct(result.mdd * 100, 1);
    }

    function renderPlot(prefixLength = 1) {
        const result = simulate();

        updateControlsText(result);

        const safePrefixLength = Math.max(
            1,
            Math.min(prefixLength, result.wealth.length)
        );

        const visibleDates = payload.dates.slice(0, safePrefixLength);
        const visibleWealth = result.wealth.slice(0, safePrefixLength);

        const barY = [
            result.winContribution,
            result.lossContribution,
            result.netEv
        ];

        const barText = barY.map(v => v.toFixed(2) + "%");

        const layoutUpdate = {
            "annotations[0].text": `Strategy Expected Value: EV ${result.netEv.toFixed(2)}% / trade`,
            "annotations[1].text": (
                safePrefixLength >= result.wealth.length
                    ? `Trading Wealth Path: Final $${result.finalWealth.toFixed(2)} | Win Rate ${(result.realizedWinRate * 100).toFixed(1)}% | MDD ${(result.mdd * 100).toFixed(1)}%`
                    : `Trading Wealth Path: Animating ${safePrefixLength} / ${result.wealth.length} trades`
            ),
            "yaxis.range": barYRange(barY),
            "yaxis2.range": wealthYRange(result.wealth),
            "title.text":
                "Expected Value Inputs vs. Simulated Trading Wealth Path" +
                "<br><sup>" +
                `P(win): ${result.pWinPct.toFixed(0)}% · ` +
                `Win Size: ${result.winSizePct.toFixed(2)}% · ` +
                `Loss Size: ${result.lossSizePct.toFixed(2)}% · ` +
                `EV: ${result.netEv.toFixed(2)}% / trade` +
                "</sup>"
        };

        Plotly.restyle(
            plotDiv,
            {
                y: [barY],
                text: [barText]
            },
            [0]
        );

        Plotly.restyle(
            plotDiv,
            {
                x: [visibleDates],
                y: [visibleWealth]
            },
            [1]
        );

        Plotly.relayout(plotDiv, layoutUpdate);
    }

    function stopAnimation() {
        if (animationTimer !== null) {
            clearInterval(animationTimer);
            animationTimer = null;
        }

        animateStrategyButton.textContent = "▶ Play Animation";
    }

    function animateCurrentStrategy() {
        stopAnimation();

        const result = simulate();
        let prefixLength = 1;

        renderPlot(prefixLength);

        animateStrategyButton.textContent = "⏸ Stop Animation";

        animationTimer = setInterval(() => {
            prefixLength = Math.min(
                prefixLength + payload.frame_stride,
                result.wealth.length
            );

            renderPlot(prefixLength);

            if (prefixLength >= result.wealth.length) {
                stopAnimation();
                renderPlot(result.wealth.length);
            }
        }, payload.frame_duration_ms);
    }

    function resetDefaults() {
        stopAnimation();

        winProbSlider.value = "__DEFAULT_WIN_PROBABILITY__";
        winSizeSlider.value = "__DEFAULT_WIN_SIZE__";
        lossSizeSlider.value = "__DEFAULT_LOSS_SIZE__";

        renderPlot(1);
    }

    winProbSlider.addEventListener("input", () => {
        stopAnimation();
        renderPlot(1);
    });

    winSizeSlider.addEventListener("input", () => {
        stopAnimation();
        renderPlot(1);
    });

    lossSizeSlider.addEventListener("input", () => {
        stopAnimation();
        renderPlot(1);
    });

    animateStrategyButton.addEventListener("click", () => {
        if (animationTimer !== null) {
            stopAnimation();
            renderPlot(1);
        } else {
            animateCurrentStrategy();
        }
    });

    showFullPathButton.addEventListener("click", () => {
        stopAnimation();
        const result = simulate();
        renderPlot(result.wealth.length);
    });

    resetStrategyButton.addEventListener("click", resetDefaults);

    renderPlot(1);
})();
</script>
"""

script_html = (
    script_template
    .replace("__PAYLOAD_JSON__", payload_json)
    .replace("__DIV_ID__", div_id)
    .replace("__DEFAULT_WIN_PROBABILITY__", str(DEFAULT_WIN_PROBABILITY))
    .replace("__DEFAULT_WIN_SIZE__", str(DEFAULT_WIN_SIZE))
    .replace("__DEFAULT_LOSS_SIZE__", str(DEFAULT_LOSS_SIZE))
)

full_html = controls_html + plot_html + script_html

display(HTML(full_html))

# Optional export:
with open("ev_strategy_slider_animation_fixed.html", "w", encoding="utf-8") as f:
    f.write(full_html)

###### ______________________________________________________________________________________________________________________________________

##### Probabilities in the Markets Don't Converge

Even with the framework above we need to understand that reality is uncertain and distributions, probabilities, and statistics change

There is no gaurenteed convergence (even conditioning on market regime) should we observe the same buy indicator many times...

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

START_DATE = "2025-01-01"
END_DATE = "2026-12-31"

SEED = 42

LEFT_PATH_NAME = "Coin Flip LLN"
RIGHT_PATH_NAME = "Non-Stationary Strategy Win Probability"

COIN_TRUE_EXPECTATION = 0.50

# Animation settings
FRAME_STRIDE = 5
FRAME_DURATION = 20

# Figure settings
FIG_WIDTH = 1200
FIG_HEIGHT = 650

# ============================================================
# Generate synthetic trading dates
# ============================================================

dates = pd.bdate_range(START_DATE, END_DATE)

if len(dates) < 2:
    raise ValueError("Need at least two business days to create an animated path.")

rng = np.random.default_rng(SEED)
n_days = len(dates)

# ============================================================
# Left chart: stationary coin flip LLN
# ============================================================

coin_flips = rng.random(n_days) < COIN_TRUE_EXPECTATION
coin_values = coin_flips.astype(float)

# Running estimate of E[Heads]
coin_running_expectation = np.cumsum(coin_values) / np.arange(1, n_days + 1)

coin_final_estimate = coin_running_expectation[-1]
coin_final_error = coin_final_estimate - COIN_TRUE_EXPECTATION

# ============================================================
# Right chart: non-stationary strategy win probability
# ============================================================

# Regime boundaries
regime_1_end = int(n_days * 0.38)
regime_2_end = int(n_days * 0.70)

# This is the non-stationarity:
# The true probability of winning a trade changes across regimes.
strategy_true_win_probability = np.zeros(n_days)

# Regime 1: strategy appears strong
strategy_true_win_probability[:regime_1_end] = 0.58

# Regime 2: edge decays / crowding / market changes
strategy_true_win_probability[regime_1_end:regime_2_end] = 0.42

# Regime 3: partial recovery, but not the original edge
strategy_true_win_probability[regime_2_end:] = 0.49

# Add a small smooth wobble so the "true" process is not perfectly blocky,
# while keeping the regime interpretation clear.
t = np.linspace(0, 1, n_days)
strategy_true_win_probability += 0.018 * np.sin(7 * np.pi * t)
strategy_true_win_probability = np.clip(strategy_true_win_probability, 0.35, 0.65)

# Realized trade wins from the time-varying probability process
strategy_wins = rng.random(n_days) < strategy_true_win_probability
strategy_win_values = strategy_wins.astype(float)

# Running historical win rate.
# This is the misleading LLN-like estimate when the data-generating process changes.
strategy_running_win_rate = np.cumsum(strategy_win_values) / np.arange(1, n_days + 1)

strategy_final_running_win_rate = strategy_running_win_rate[-1]
strategy_final_true_probability = strategy_true_win_probability[-1]
strategy_average_true_probability = np.mean(strategy_true_win_probability)

# ============================================================
# Build DataFrame
# ============================================================

df = pd.DataFrame({
    "date": dates,
    "coin_running_expectation": coin_running_expectation,
    "strategy_running_win_rate": strategy_running_win_rate,
    "strategy_true_win_probability": strategy_true_win_probability
})

latest_date = df["date"].iloc[-1]

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

coin_color = "#00ff88"
coin_expectation_color = "#ffffff"

strategy_running_color = "#ffaa33"
strategy_true_probability_color = "#ff66cc"

baseline_color = "#777777"
regime_line_color = "rgba(255,255,255,0.35)"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# Y-axis ranges
LEFT_Y_RANGE = [0, 1]
RIGHT_Y_RANGE = [0, 1]

# Convert regime dates to native datetimes for Plotly shapes
regime_1_end_x = pd.Timestamp(dates[regime_1_end]).to_pydatetime()
regime_2_end_x = pd.Timestamp(dates[regime_2_end]).to_pydatetime()

# In-chart legend placement
left_legend_x = dates[int(n_days * 0.05)]
right_legend_x = dates[int(n_days * 0.05)]

left_legend_y_top = 0.92
left_legend_y_second = 0.84

right_legend_y_top = 0.92
right_legend_y_third = 0.84

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(
        (
            f"{LEFT_PATH_NAME}: Running E[Heads] "
            f"→ {coin_final_estimate:.2%}"
        ),
        (
            f"{RIGHT_PATH_NAME}: "
            f"Final Running Win Rate {strategy_final_running_win_rate:.2%}"
        )
    )
)

# ============================================================
# Initial traces
# ============================================================

# Left: running coin flip expectation
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["coin_running_expectation"].iloc[0]],
        mode="lines",
        line=dict(color=coin_color, width=4),
        name="Running E[Heads]",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Running E[Heads]: %{y:.2%}<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=1
)

# Right: running historical win rate
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["strategy_running_win_rate"].iloc[0]],
        mode="lines",
        line=dict(color=strategy_running_color, width=4),
        name="Running Win Rate",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Running Win Rate: %{y:.2%}<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=2
)

# Right: moving true probability of winning
fig.add_trace(
    go.Scatter(
        x=[df["date"].iloc[0]],
        y=[df["strategy_true_win_probability"].iloc[0]],
        mode="lines",
        line=dict(color=strategy_true_probability_color, width=3, dash="dash"),
        name="Moving True P(win)",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Moving True P(win): %{y:.2%}<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=2
)

# Trace indices
COIN_TRACE_INDEX = 0
STRATEGY_RUNNING_TRACE_INDEX = 1
STRATEGY_TRUE_PROB_TRACE_INDEX = 2

# ============================================================
# Static reference lines and regime markers
# ============================================================

# Left: visible 50% theoretical reference line.
# This is intentionally drawn as a layout shape above the traces so it remains visible.
fig.add_shape(
    type="line",
    x0=pd.Timestamp(dates[0]).to_pydatetime(),
    x1=pd.Timestamp(dates[-1]).to_pydatetime(),
    y0=COIN_TRUE_EXPECTATION,
    y1=COIN_TRUE_EXPECTATION,
    line=dict(color=coin_expectation_color, width=3, dash="dash"),
    opacity=0.95,
    layer="above",
    row=1,
    col=1
)

# Right: 50% break-even probability reference line
fig.add_shape(
    type="line",
    x0=pd.Timestamp(dates[0]).to_pydatetime(),
    x1=pd.Timestamp(dates[-1]).to_pydatetime(),
    y0=0.50,
    y1=0.50,
    line=dict(color=baseline_color, width=1.5, dash="dash"),
    opacity=0.70,
    layer="above",
    row=1,
    col=2
)

# Right: regime shift markers
for regime_x in [regime_1_end_x, regime_2_end_x]:
    fig.add_shape(
        type="line",
        x0=regime_x,
        x1=regime_x,
        y0=RIGHT_Y_RANGE[0],
        y1=RIGHT_Y_RANGE[1],
        line=dict(color=regime_line_color, width=1.25, dash="dot"),
        opacity=0.80,
        layer="above",
        row=1,
        col=2
    )

fig.add_annotation(
    x=regime_1_end_x,
    y=RIGHT_Y_RANGE[1],
    text="Regime shift",
    showarrow=False,
    font=dict(color=off_white, size=11),
    bgcolor="rgba(0,0,0,0.30)",
    xanchor="left",
    yanchor="top",
    row=1,
    col=2
)

fig.add_annotation(
    x=regime_2_end_x,
    y=RIGHT_Y_RANGE[1],
    text="Regime shift",
    showarrow=False,
    font=dict(color=off_white, size=11),
    bgcolor="rgba(0,0,0,0.30)",
    xanchor="left",
    yanchor="top",
    row=1,
    col=2
)

# ============================================================
# In-chart legend labels
# ============================================================

# Left chart legend
fig.add_annotation(
    x=left_legend_x,
    y=left_legend_y_top,
    text="━ Running E[Heads]",
    showarrow=False,
    font=dict(color=coin_color, size=13),
    bgcolor="rgba(0,0,0,0.35)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="middle",
    row=1,
    col=1
)

fig.add_annotation(
    x=left_legend_x,
    y=left_legend_y_second,
    text="┄ 50% Theoretical Line",
    showarrow=False,
    font=dict(color=coin_expectation_color, size=13),
    bgcolor="rgba(0,0,0,0.35)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="middle",
    row=1,
    col=1
)

# Right chart legend
fig.add_annotation(
    x=right_legend_x,
    y=right_legend_y_top,
    text="━ Running Win Rate",
    showarrow=False,
    font=dict(color=strategy_running_color, size=13),
    bgcolor="rgba(0,0,0,0.35)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="middle",
    row=1,
    col=2
)

fig.add_annotation(
    x=right_legend_x,
    y=right_legend_y_third,
    text="┄ Moving True P(win)",
    showarrow=False,
    font=dict(color=strategy_true_probability_color, size=13),
    bgcolor="rgba(0,0,0,0.35)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="middle",
    row=1,
    col=2
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(1, len(df), FRAME_STRIDE))

if not frame_indices or frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    plot_slice = df.iloc[:i + 1]

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=plot_slice["date"],
                    y=plot_slice["coin_running_expectation"]
                ),
                go.Scatter(
                    x=plot_slice["date"],
                    y=plot_slice["strategy_running_win_rate"]
                ),
                go.Scatter(
                    x=plot_slice["date"],
                    y=plot_slice["strategy_true_win_probability"]
                )
            ],
            traces=[
                COIN_TRACE_INDEX,
                STRATEGY_RUNNING_TRACE_INDEX,
                STRATEGY_TRUE_PROB_TRACE_INDEX
            ],
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["date"].iloc[i].strftime("%b %Y"),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "Classical LLN vs. Non-Stationary Strategy Win Probability"
            "<br><sup>"
            f"{START_DATE} to {latest_date:%Y-%m-%d} · "
            f"Coin Flip Running E[Heads]: {coin_final_estimate:.2%} "
            f"(Error: {coin_final_error:+.2%}) · "
            f"Strategy Running Win Rate: {strategy_final_running_win_rate:.2%} · "
            f"Current True P(win): {strategy_final_true_probability:.2%} · "
            f"Avg True P(win): {strategy_average_true_probability:.2%}"
            "</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=FIG_HEIGHT,
    width=FIG_WIDTH,
    margin=dict(t=120, b=115, r=50, l=75),
    showlegend=False,
    hovermode="closest",

    # Buttons closer to the bottom of the charts.
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": False
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {
                            "duration": 0,
                            "redraw": False
                        },
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 46},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": -0.08,
        "yanchor": "top"
    }],

    # Slider closer to the bottom of the charts.
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 18},
        "len": 0.85,
        "x": 0.15,
        "y": -0.08,
        "steps": slider_steps
    }]
)

# ============================================================
# Axes
# ============================================================

# Left subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=LEFT_Y_RANGE,
    title_text="Running Estimate of E[Heads]",
    tickformat=".0%"
)

# Right subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[df["date"].iloc[0], df["date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=RIGHT_Y_RANGE,
    title_text="Probability of Winning a Trade",
    tickformat=".0%"
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("lln_coin_flip_vs_nonstationary_strategy_win_probability.html", include_plotlyjs="cdn")

Interestingly, we don't need convergence or for our models to be "correct" to make money.

Understanding the underlying math, probability statistics, and finance at play and this notion of model and parameter risk is how money is made.

###### ______________________________________________________________________________________________________________________________________

##### **Reality:** Everything is a bet on Positioning and Survival, not Exclusively Profits 

Regimes dictate strategy and portfolio composition, prioritizing positioning and survival above all else.

Profits mean nothing if you blowout your account.

If that sounds boring, you're right, it is!  That's why you want the magical indicator. 

##### 2021 Market Returns vs Hedged Portfolio Returns

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

CSV_PATH = "spy_kmlm_prices_2021_2022.csv"

# Fallback if your local file is named without ".csv"
if not os.path.exists(CSV_PATH) and os.path.exists("spy_kmlm_prices_2021_2022"):
    CSV_PATH = "spy_kmlm_prices_2021_2022"

START_DATE = "2021-01-01"
END_DATE = "2021-12-31"

INITIAL_WEALTH = 100

LEFT_PATH_NAME = "Unhedged SPY"
RIGHT_PATH_NAME = "50/50 SPY + KMLM"

# Portfolio weights
SPY_ONLY_WEIGHT = 1.00

HEDGED_SPY_WEIGHT = 0.50
HEDGED_KMLM_WEIGHT = 0.50

# Animation settings
FRAME_STRIDE = 3
FRAME_DURATION = 20

# Figure settings
FIG_WIDTH = 1200
FIG_HEIGHT = 650

# ============================================================
# Load CSV
# ============================================================

df = pd.read_csv(CSV_PATH)

# Normalize column names defensively
df.columns = [c.strip() for c in df.columns]

required_columns = {"Date", "SPY", "KMLM"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"CSV is missing required columns: {missing_columns}")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df = df[
    (df["Date"] >= pd.Timestamp(START_DATE)) &
    (df["Date"] <= pd.Timestamp(END_DATE))
].copy()

if df.empty:
    raise ValueError("No data found in the requested date range.")

# ============================================================
# Return construction
# ============================================================

df["spy_return"] = df["SPY"].pct_change().fillna(0)
df["kmlm_return"] = df["KMLM"].pct_change().fillna(0)

df["spy_only_return"] = SPY_ONLY_WEIGHT * df["spy_return"]

df["hedged_50_50_return"] = (
    HEDGED_SPY_WEIGHT * df["spy_return"]
    + HEDGED_KMLM_WEIGHT * df["kmlm_return"]
)

df["spy_only_wealth"] = INITIAL_WEALTH * (1 + df["spy_only_return"]).cumprod()
df["hedged_50_50_wealth"] = INITIAL_WEALTH * (1 + df["hedged_50_50_return"]).cumprod()

df["spy_only_cum_return"] = df["spy_only_wealth"] / INITIAL_WEALTH - 1
df["hedged_50_50_cum_return"] = df["hedged_50_50_wealth"] / INITIAL_WEALTH - 1

# ============================================================
# Summary stats
# ============================================================

latest_date = df["Date"].iloc[-1]

def annualized_cagr(final_wealth, initial_wealth, n_days):
    if n_days <= 1:
        return np.nan

    years = n_days / 252

    if final_wealth <= 0:
        return np.nan

    return (final_wealth / initial_wealth) ** (1 / years) - 1


def annualized_vol(simple_returns):
    simple_returns = np.asarray(simple_returns)

    if len(simple_returns) < 2:
        return np.nan

    return np.std(simple_returns, ddof=1) * np.sqrt(252)


def annualized_sharpe(simple_returns):
    simple_returns = np.asarray(simple_returns)

    if len(simple_returns) < 2:
        return np.nan

    mean = np.mean(simple_returns)
    std = np.std(simple_returns, ddof=1)

    if std == 0:
        return np.nan

    return mean / std * np.sqrt(252)


def max_drawdown_from_wealth(wealth):
    wealth = np.asarray(wealth)
    running_max = np.maximum.accumulate(wealth)
    drawdown = 1 - wealth / running_max
    return np.max(drawdown)


spy_final_wealth = df["spy_only_wealth"].iloc[-1]
hedged_final_wealth = df["hedged_50_50_wealth"].iloc[-1]

spy_final_return = df["spy_only_cum_return"].iloc[-1]
hedged_final_return = df["hedged_50_50_cum_return"].iloc[-1]
return_spread = hedged_final_return - spy_final_return

spy_cagr = annualized_cagr(spy_final_wealth, INITIAL_WEALTH, len(df))
hedged_cagr = annualized_cagr(hedged_final_wealth, INITIAL_WEALTH, len(df))

spy_vol = annualized_vol(df["spy_only_return"])
hedged_vol = annualized_vol(df["hedged_50_50_return"])

spy_sharpe = annualized_sharpe(df["spy_only_return"])
hedged_sharpe = annualized_sharpe(df["hedged_50_50_return"])

spy_mdd = max_drawdown_from_wealth(df["spy_only_wealth"])
hedged_mdd = max_drawdown_from_wealth(df["hedged_50_50_wealth"])

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

spy_color = "#ffaa33"
hedged_color = "#00ff88"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# Shared y-axis range across both charts
wealth_min = min(
    df["spy_only_wealth"].min(),
    df["hedged_50_50_wealth"].min()
)

wealth_max = max(
    df["spy_only_wealth"].max(),
    df["hedged_50_50_wealth"].max()
)

y_pad = max((wealth_max - wealth_min) * 0.12, 5)

Y_RANGE = [
    max(0, wealth_min - y_pad),
    wealth_max + y_pad
]

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(
        (
            f"{LEFT_PATH_NAME}: "
            f"CAGR {spy_cagr:.2%} | "
            f"Sharpe {spy_sharpe:.2f} | "
            f"MDD {spy_mdd:.1%}"
        ),
        (
            f"{RIGHT_PATH_NAME}: "
            f"CAGR {hedged_cagr:.2%} | "
            f"Sharpe {hedged_sharpe:.2f} | "
            f"MDD {hedged_mdd:.1%}"
        )
    )
)

# Starting wealth baselines
fig.add_hline(
    y=INITIAL_WEALTH,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1
)

fig.add_hline(
    y=INITIAL_WEALTH,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2
)

# Left: Unhedged SPY
fig.add_trace(
    go.Scatter(
        x=[df["Date"].iloc[0]],
        y=[df["spy_only_wealth"].iloc[0]],
        mode="lines",
        line=dict(color=spy_color, width=4),
        name=LEFT_PATH_NAME,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{LEFT_PATH_NAME}: $%{{y:.2f}}<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=1
)

# Right: 50/50 SPY + KMLM
fig.add_trace(
    go.Scatter(
        x=[df["Date"].iloc[0]],
        y=[df["hedged_50_50_wealth"].iloc[0]],
        mode="lines",
        line=dict(color=hedged_color, width=4),
        name=RIGHT_PATH_NAME,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{RIGHT_PATH_NAME}: $%{{y:.2f}}<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=2
)

# ============================================================
# In-chart labels
# ============================================================

left_label_x = df["Date"].iloc[int(len(df) * 0.04)]
right_label_x = df["Date"].iloc[int(len(df) * 0.04)]

label_y = Y_RANGE[1] - (Y_RANGE[1] - Y_RANGE[0]) * 0.12

fig.add_annotation(
    x=left_label_x,
    y=label_y,
    text=(
        f"━ {LEFT_PATH_NAME}<br>"
        f"Final: ${spy_final_wealth:.2f}<br>"
        f"Return: {spy_final_return:.1%}"
    ),
    showarrow=False,
    font=dict(color=spy_color, size=13),
    align="left",
    bgcolor="rgba(0,0,0,0.38)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="top",
    row=1,
    col=1
)

fig.add_annotation(
    x=right_label_x,
    y=label_y,
    text=(
        f"━ {RIGHT_PATH_NAME}<br>"
        f"Final: ${hedged_final_wealth:.2f}<br>"
        f"Return: {hedged_final_return:.1%}"
    ),
    showarrow=False,
    font=dict(color=hedged_color, size=13),
    align="left",
    bgcolor="rgba(0,0,0,0.38)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="top",
    row=1,
    col=2
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(1, len(df), FRAME_STRIDE))

if not frame_indices or frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    plot_slice = df.iloc[:i + 1]

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=plot_slice["Date"],
                    y=plot_slice["spy_only_wealth"]
                ),
                go.Scatter(
                    x=plot_slice["Date"],
                    y=plot_slice["hedged_50_50_wealth"]
                )
            ],
            traces=[0, 1],
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["Date"].iloc[i].strftime("%b %Y"),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "2021 Portfolio Comparison: Unhedged SPY vs. 50/50 SPY + KMLM"
            "<br><sup>"
            f"{START_DATE} to {latest_date:%Y-%m-%d} · "
            f"{LEFT_PATH_NAME} Final Wealth: ${spy_final_wealth:.2f} · "
            f"{RIGHT_PATH_NAME} Final Wealth: ${hedged_final_wealth:.2f} · "
            f"Return Spread: {return_spread:.1%}"
            "</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=FIG_HEIGHT,
    width=FIG_WIDTH,
    margin=dict(t=120, b=150, r=50, l=75),
    showlegend=False,
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": False
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {
                            "duration": 0,
                            "redraw": False
                        },
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_annotations(font=dict(color=off_white, size=14))

# ============================================================
# Axes
# ============================================================

# Left subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["Date"].iloc[0], df["Date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=Y_RANGE,
    title_text="Portfolio Value, Starting at $100",
    tickprefix="$",
    tickformat=",.0f"
)

# Right subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[df["Date"].iloc[0], df["Date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=Y_RANGE,
    title_text="Portfolio Value, Starting at $100",
    tickprefix="$",
    tickformat=",.0f"
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("spy_vs_50_50_spy_kmlm_2021_animated.html", include_plotlyjs="cdn")

##### 2022 Market Returns vs Hedged Portfolio Returns

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

CSV_PATH = "spy_kmlm_prices_2021_2022.csv"

# Fallback if your local file is named without ".csv"
if not os.path.exists(CSV_PATH) and os.path.exists("spy_kmlm_prices_2021_2022"):
    CSV_PATH = "spy_kmlm_prices_2021_2022"

START_DATE = "2022-01-01"
END_DATE = "2022-12-31"

INITIAL_WEALTH = 100

LEFT_PATH_NAME = "Unhedged SPY"
RIGHT_PATH_NAME = "50/50 SPY + KMLM"

# Portfolio weights
SPY_ONLY_WEIGHT = 1.00

HEDGED_SPY_WEIGHT = 0.50
HEDGED_KMLM_WEIGHT = 0.50

# Animation settings
FRAME_STRIDE = 3
FRAME_DURATION = 20

# Figure settings
FIG_WIDTH = 1200
FIG_HEIGHT = 650

# ============================================================
# Load CSV
# ============================================================

df = pd.read_csv(CSV_PATH)

# Normalize column names defensively
df.columns = [c.strip() for c in df.columns]

required_columns = {"Date", "SPY", "KMLM"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"CSV is missing required columns: {missing_columns}")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df = df[
    (df["Date"] >= pd.Timestamp(START_DATE)) &
    (df["Date"] <= pd.Timestamp(END_DATE))
].copy()

if df.empty:
    raise ValueError("No data found in the requested date range.")

# ============================================================
# Return construction
# ============================================================

df["spy_return"] = df["SPY"].pct_change().fillna(0)
df["kmlm_return"] = df["KMLM"].pct_change().fillna(0)

df["spy_only_return"] = SPY_ONLY_WEIGHT * df["spy_return"]

df["hedged_50_50_return"] = (
    HEDGED_SPY_WEIGHT * df["spy_return"]
    + HEDGED_KMLM_WEIGHT * df["kmlm_return"]
)

df["spy_only_wealth"] = INITIAL_WEALTH * (1 + df["spy_only_return"]).cumprod()
df["hedged_50_50_wealth"] = INITIAL_WEALTH * (1 + df["hedged_50_50_return"]).cumprod()

df["spy_only_cum_return"] = df["spy_only_wealth"] / INITIAL_WEALTH - 1
df["hedged_50_50_cum_return"] = df["hedged_50_50_wealth"] / INITIAL_WEALTH - 1

# ============================================================
# Summary stats
# ============================================================

latest_date = df["Date"].iloc[-1]

def annualized_cagr(final_wealth, initial_wealth, n_days):
    if n_days <= 1:
        return np.nan

    years = n_days / 252

    if final_wealth <= 0:
        return np.nan

    return (final_wealth / initial_wealth) ** (1 / years) - 1


def annualized_vol(simple_returns):
    simple_returns = np.asarray(simple_returns)

    if len(simple_returns) < 2:
        return np.nan

    return np.std(simple_returns, ddof=1) * np.sqrt(252)


def annualized_sharpe(simple_returns):
    simple_returns = np.asarray(simple_returns)

    if len(simple_returns) < 2:
        return np.nan

    mean = np.mean(simple_returns)
    std = np.std(simple_returns, ddof=1)

    if std == 0:
        return np.nan

    return mean / std * np.sqrt(252)


def max_drawdown_from_wealth(wealth):
    wealth = np.asarray(wealth)
    running_max = np.maximum.accumulate(wealth)
    drawdown = 1 - wealth / running_max
    return np.max(drawdown)


spy_final_wealth = df["spy_only_wealth"].iloc[-1]
hedged_final_wealth = df["hedged_50_50_wealth"].iloc[-1]

spy_final_return = df["spy_only_cum_return"].iloc[-1]
hedged_final_return = df["hedged_50_50_cum_return"].iloc[-1]
return_spread = hedged_final_return - spy_final_return

spy_cagr = annualized_cagr(spy_final_wealth, INITIAL_WEALTH, len(df))
hedged_cagr = annualized_cagr(hedged_final_wealth, INITIAL_WEALTH, len(df))

spy_vol = annualized_vol(df["spy_only_return"])
hedged_vol = annualized_vol(df["hedged_50_50_return"])

spy_sharpe = annualized_sharpe(df["spy_only_return"])
hedged_sharpe = annualized_sharpe(df["hedged_50_50_return"])

spy_mdd = max_drawdown_from_wealth(df["spy_only_wealth"])
hedged_mdd = max_drawdown_from_wealth(df["hedged_50_50_wealth"])

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

spy_color = "#ffaa33"
hedged_color = "#00ff88"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# Shared y-axis range across both charts
wealth_min = min(
    df["spy_only_wealth"].min(),
    df["hedged_50_50_wealth"].min()
)

wealth_max = max(
    df["spy_only_wealth"].max(),
    df["hedged_50_50_wealth"].max()
)

y_pad = max((wealth_max - wealth_min) * 0.12, 5)

Y_RANGE = [
    max(0, wealth_min - y_pad),
    wealth_max + y_pad
]

# For the right subplot (second chart), override Y_RANGE with [95, 105]
Y_RANGE_RIGHT = [95, 105]

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(
        (
            f"{LEFT_PATH_NAME}: "
            f"CAGR {spy_cagr:.2%} | "
            f"Sharpe {spy_sharpe:.2f} | "
            f"MDD {spy_mdd:.1%}"
        ),
        (
            f"{RIGHT_PATH_NAME}: "
            f"CAGR {hedged_cagr:.2%} | "
            f"Sharpe {hedged_sharpe:.2f} | "
            f"MDD {hedged_mdd:.1%}"
        )
    )
)

# Starting wealth baselines
fig.add_hline(
    y=INITIAL_WEALTH,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1
)

fig.add_hline(
    y=INITIAL_WEALTH,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2
)

# Left: Unhedged SPY
fig.add_trace(
    go.Scatter(
        x=[df["Date"].iloc[0]],
        y=[df["spy_only_wealth"].iloc[0]],
        mode="lines",
        line=dict(color=spy_color, width=4),
        name=LEFT_PATH_NAME,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{LEFT_PATH_NAME}: $%{{y:.2f}}<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=1
)

# Right: 50/50 SPY + KMLM
fig.add_trace(
    go.Scatter(
        x=[df["Date"].iloc[0]],
        y=[df["hedged_50_50_wealth"].iloc[0]],
        mode="lines",
        line=dict(color=hedged_color, width=4),
        name=RIGHT_PATH_NAME,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{RIGHT_PATH_NAME}: $%{{y:.2f}}<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=2
)

# ============================================================
# In-chart labels
# ============================================================

left_label_x = df["Date"].iloc[int(len(df) * 0.04)]
right_label_x = df["Date"].iloc[int(len(df) * 0.04)]

label_y = Y_RANGE[1] - (Y_RANGE[1] - Y_RANGE[0]) * 0.12

fig.add_annotation(
    x=left_label_x,
    y=label_y,
    text=(
        f"━ {LEFT_PATH_NAME}<br>"
        f"Final: ${spy_final_wealth:.2f}<br>"
        f"Return: {spy_final_return:.1%}"
    ),
    showarrow=False,
    font=dict(color=spy_color, size=13),
    align="left",
    bgcolor="rgba(0,0,0,0.38)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="top",
    row=1,
    col=1
)

fig.add_annotation(
    x=right_label_x,
    y=label_y,
    text=(
        f"━ {RIGHT_PATH_NAME}<br>"
        f"Final: ${hedged_final_wealth:.2f}<br>"
        f"Return: {hedged_final_return:.1%}"
    ),
    showarrow=False,
    font=dict(color=hedged_color, size=13),
    align="left",
    bgcolor="rgba(0,0,0,0.38)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="top",
    row=1,
    col=2
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(1, len(df), FRAME_STRIDE))

if not frame_indices or frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    plot_slice = df.iloc[:i + 1]

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=plot_slice["Date"],
                    y=plot_slice["spy_only_wealth"]
                ),
                go.Scatter(
                    x=plot_slice["Date"],
                    y=plot_slice["hedged_50_50_wealth"]
                )
            ],
            traces=[0, 1],
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["Date"].iloc[i].strftime("%b %Y"),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "2022 Portfolio Comparison: Unhedged SPY vs. 50/50 SPY + KMLM"
            "<br><sup>"
            f"{START_DATE} to {latest_date:%Y-%m-%d} · "
            f"{LEFT_PATH_NAME} Final Wealth: ${spy_final_wealth:.2f} · "
            f"{RIGHT_PATH_NAME} Final Wealth: ${hedged_final_wealth:.2f} · "
            f"Return Spread: {return_spread:.1%}"
            "</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=FIG_HEIGHT,
    width=FIG_WIDTH,
    margin=dict(t=120, b=150, r=50, l=75),
    showlegend=False,
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": False
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {
                            "duration": 0,
                            "redraw": False
                        },
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_annotations(font=dict(color=off_white, size=14))

# ============================================================
# Axes
# ============================================================

# Left subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["Date"].iloc[0], df["Date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=Y_RANGE,
    title_text="Portfolio Value, Starting at $100",
    tickprefix="$",
    tickformat=",.0f"
)

# Right subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[df["Date"].iloc[0], df["Date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=Y_RANGE_RIGHT,
    title_text="Portfolio Value, Starting at $100",
    tickprefix="$",
    tickformat=",.0f"
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("spy_vs_50_50_spy_kmlm_2022_animated.html", include_plotlyjs="cdn")

##### Full Sample Market Returns vs Hedged Portfolio Returns

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

CSV_PATH = "spy_kmlm_prices_2021_2022.csv"

# Fallback if your local file is named without ".csv"
if not os.path.exists(CSV_PATH) and os.path.exists("spy_kmlm_prices_2021_2022"):
    CSV_PATH = "spy_kmlm_prices_2021_2022"

START_DATE = "2021-01-01"
END_DATE = "2022-12-31"

INITIAL_WEALTH = 100

LEFT_PATH_NAME = "Unhedged SPY"
RIGHT_PATH_NAME = "50/50 SPY + KMLM"

# Portfolio weights
SPY_ONLY_WEIGHT = 1.00

HEDGED_SPY_WEIGHT = 0.50
HEDGED_KMLM_WEIGHT = 0.50

# Animation settings
FRAME_STRIDE = 5
FRAME_DURATION = 20

# Figure settings
FIG_WIDTH = 1200
FIG_HEIGHT = 650

# ============================================================
# Load CSV
# ============================================================

df = pd.read_csv(CSV_PATH)

# Normalize column names defensively
df.columns = [c.strip() for c in df.columns]

required_columns = {"Date", "SPY", "KMLM"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"CSV is missing required columns: {missing_columns}")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df = df[
    (df["Date"] >= pd.Timestamp(START_DATE)) &
    (df["Date"] <= pd.Timestamp(END_DATE))
].copy()

if df.empty:
    raise ValueError("No data found in the requested date range.")

# ============================================================
# Return construction
# ============================================================

df["spy_return"] = df["SPY"].pct_change().fillna(0)
df["kmlm_return"] = df["KMLM"].pct_change().fillna(0)

df["spy_only_return"] = SPY_ONLY_WEIGHT * df["spy_return"]

df["hedged_50_50_return"] = (
    HEDGED_SPY_WEIGHT * df["spy_return"]
    + HEDGED_KMLM_WEIGHT * df["kmlm_return"]
)

df["spy_only_wealth"] = INITIAL_WEALTH * (1 + df["spy_only_return"]).cumprod()
df["hedged_50_50_wealth"] = INITIAL_WEALTH * (1 + df["hedged_50_50_return"]).cumprod()

df["spy_only_cum_return"] = df["spy_only_wealth"] / INITIAL_WEALTH - 1
df["hedged_50_50_cum_return"] = df["hedged_50_50_wealth"] / INITIAL_WEALTH - 1

# ============================================================
# Summary stats
# ============================================================

latest_date = df["Date"].iloc[-1]

def annualized_cagr(final_wealth, initial_wealth, n_days):
    if n_days <= 1:
        return np.nan

    years = n_days / 252

    if final_wealth <= 0:
        return np.nan

    return (final_wealth / initial_wealth) ** (1 / years) - 1


def annualized_vol(simple_returns):
    simple_returns = np.asarray(simple_returns)

    if len(simple_returns) < 2:
        return np.nan

    return np.std(simple_returns, ddof=1) * np.sqrt(252)


def annualized_sharpe(simple_returns):
    simple_returns = np.asarray(simple_returns)

    if len(simple_returns) < 2:
        return np.nan

    mean = np.mean(simple_returns)
    std = np.std(simple_returns, ddof=1)

    if std == 0:
        return np.nan

    return mean / std * np.sqrt(252)


def max_drawdown_from_wealth(wealth):
    wealth = np.asarray(wealth)
    running_max = np.maximum.accumulate(wealth)
    drawdown = 1 - wealth / running_max
    return np.max(drawdown)


spy_final_wealth = df["spy_only_wealth"].iloc[-1]
hedged_final_wealth = df["hedged_50_50_wealth"].iloc[-1]

spy_final_return = df["spy_only_cum_return"].iloc[-1]
hedged_final_return = df["hedged_50_50_cum_return"].iloc[-1]
return_spread = hedged_final_return - spy_final_return

spy_cagr = annualized_cagr(spy_final_wealth, INITIAL_WEALTH, len(df))
hedged_cagr = annualized_cagr(hedged_final_wealth, INITIAL_WEALTH, len(df))

spy_vol = annualized_vol(df["spy_only_return"])
hedged_vol = annualized_vol(df["hedged_50_50_return"])

spy_sharpe = annualized_sharpe(df["spy_only_return"])
hedged_sharpe = annualized_sharpe(df["hedged_50_50_return"])

spy_mdd = max_drawdown_from_wealth(df["spy_only_wealth"])
hedged_mdd = max_drawdown_from_wealth(df["hedged_50_50_wealth"])

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

spy_color = "#ffaa33"
hedged_color = "#00ff88"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# Shared y-axis range across both charts
wealth_min = min(
    df["spy_only_wealth"].min(),
    df["hedged_50_50_wealth"].min()
)

wealth_max = max(
    df["spy_only_wealth"].max(),
    df["hedged_50_50_wealth"].max()
)

y_pad = max((wealth_max - wealth_min) * 0.12, 5)

Y_RANGE = [
    max(0, wealth_min - y_pad),
    wealth_max + y_pad
]

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(
        (
            f"{LEFT_PATH_NAME}: "
            f"CAGR {spy_cagr:.2%} | "
            f"Sharpe {spy_sharpe:.2f} | "
            f"MDD {spy_mdd:.1%}"
        ),
        (
            f"{RIGHT_PATH_NAME}: "
            f"CAGR {hedged_cagr:.2%} | "
            f"Sharpe {hedged_sharpe:.2f} | "
            f"MDD {hedged_mdd:.1%}"
        )
    )
)

# Starting wealth baselines
fig.add_hline(
    y=INITIAL_WEALTH,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=1
)

fig.add_hline(
    y=INITIAL_WEALTH,
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    row=1,
    col=2
)

# Left: Unhedged SPY
fig.add_trace(
    go.Scatter(
        x=[df["Date"].iloc[0]],
        y=[df["spy_only_wealth"].iloc[0]],
        mode="lines",
        line=dict(color=spy_color, width=4),
        name=LEFT_PATH_NAME,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{LEFT_PATH_NAME}: $%{{y:.2f}}<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=1
)

# Right: 50/50 SPY + KMLM
fig.add_trace(
    go.Scatter(
        x=[df["Date"].iloc[0]],
        y=[df["hedged_50_50_wealth"].iloc[0]],
        mode="lines",
        line=dict(color=hedged_color, width=4),
        name=RIGHT_PATH_NAME,
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{RIGHT_PATH_NAME}: $%{{y:.2f}}<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=2
)

# ============================================================
# In-chart labels
# ============================================================

left_label_x = df["Date"].iloc[int(len(df) * 0.04)]
right_label_x = df["Date"].iloc[int(len(df) * 0.04)]

label_y = Y_RANGE[1] - (Y_RANGE[1] - Y_RANGE[0]) * 0.12

fig.add_annotation(
    x=left_label_x,
    y=label_y,
    text=(
        f"━ {LEFT_PATH_NAME}<br>"
        f"Final: ${spy_final_wealth:.2f}<br>"
        f"Return: {spy_final_return:.1%}"
    ),
    showarrow=False,
    font=dict(color=spy_color, size=13),
    align="left",
    bgcolor="rgba(0,0,0,0.38)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="top",
    row=1,
    col=1
)

fig.add_annotation(
    x=right_label_x,
    y=label_y,
    text=(
        f"━ {RIGHT_PATH_NAME}<br>"
        f"Final: ${hedged_final_wealth:.2f}<br>"
        f"Return: {hedged_final_return:.1%}"
    ),
    showarrow=False,
    font=dict(color=hedged_color, size=13),
    align="left",
    bgcolor="rgba(0,0,0,0.38)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="top",
    row=1,
    col=2
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(1, len(df), FRAME_STRIDE))

if not frame_indices or frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    plot_slice = df.iloc[:i + 1]

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=plot_slice["Date"],
                    y=plot_slice["spy_only_wealth"]
                ),
                go.Scatter(
                    x=plot_slice["Date"],
                    y=plot_slice["hedged_50_50_wealth"]
                )
            ],
            traces=[0, 1],
            name=frame_name
        )
    )

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": df["Date"].iloc[i].strftime("%b %Y"),
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "Full 2021–2022 Portfolio Comparison: Unhedged SPY vs. 50/50 SPY + KMLM"
            "<br><sup>"
            f"{START_DATE} to {latest_date:%Y-%m-%d} · "
            f"{LEFT_PATH_NAME} Final Wealth: ${spy_final_wealth:.2f} · "
            f"{RIGHT_PATH_NAME} Final Wealth: ${hedged_final_wealth:.2f} · "
            f"Return Spread: {return_spread:.1%}"
            "</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=FIG_HEIGHT,
    width=FIG_WIDTH,
    margin=dict(t=120, b=150, r=50, l=75),
    showlegend=False,
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": False
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {
                            "duration": 0,
                            "redraw": False
                        },
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_annotations(font=dict(color=off_white, size=14))

# ============================================================
# Axes
# ============================================================

# Left subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[df["Date"].iloc[0], df["Date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=Y_RANGE,
    title_text="Portfolio Value, Starting at $100",
    tickprefix="$",
    tickformat=",.0f"
)

# Right subplot
fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[df["Date"].iloc[0], df["Date"].iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=Y_RANGE,
    title_text="Portfolio Value, Starting at $100",
    tickprefix="$",
    tickformat=",.0f"
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("spy_vs_50_50_spy_kmlm_full_2021_2022_animated.html", include_plotlyjs="cdn")

###### ______________________________________________________________________________________________________________________________________

##### **Problem:** Learning is *Hard*

Math, probability and statistics, coding, quantitative finance, machine learning, artificial intelligence, the literature...

It all gets overwhelming so quickly, I've been studying it for almost 15 years and teaching it for almost 5.

It doesn't get any easier, who wants to put in that kind of work?  The payoff is delayed to an exhorbanant degree.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# Config
# ============================================================

START_YEAR = 0
END_YEAR = 20

INITIAL_PROGRESS = 0

LEFT_PATH_NAME = "The Quitter"
RIGHT_PATH_NAME = "Compounding Learner"

QUIT_YEAR = 5
EXPONENTIAL_START_YEAR = 5

# Linear / exponential settings
LINEAR_SLOPE = 2.00
EXPONENTIAL_RATE = 0.22

# Animation settings
FRAME_STRIDE = 3
FRAME_DURATION = 20

# Figure settings
FIG_WIDTH = 1200
FIG_HEIGHT = 650

# ============================================================
# Generate synthetic time axis
# ============================================================

n_points = 401
years = np.linspace(START_YEAR, END_YEAR, n_points)

if len(years) < 2:
    raise ValueError("Need at least two time points to create an animated path.")

# ============================================================
# Left chart: shallow learning curve, then quitting
# ============================================================

# Very shallow early improvement
left_pre_quit_progress = INITIAL_PROGRESS + 0.85 * np.minimum(years, QUIT_YEAR)

# After quitting, progress stops
left_progress = np.where(
    years <= QUIT_YEAR,
    left_pre_quit_progress,
    INITIAL_PROGRESS + 0.85 * QUIT_YEAR
)

left_final_progress = left_progress[-1]
left_quit_progress = INITIAL_PROGRESS + 0.85 * QUIT_YEAR

# ============================================================
# Right chart: linear first, exponential after Year 5
# ============================================================

# Linear reference line
linear_reference_progress = LINEAR_SLOPE * years

# The compounding learner tracks the linear path until Year 5
progress_at_exponential_start = LINEAR_SLOPE * EXPONENTIAL_START_YEAR

right_progress = np.where(
    years <= EXPONENTIAL_START_YEAR,
    LINEAR_SLOPE * years,
    progress_at_exponential_start
    * np.exp(EXPONENTIAL_RATE * (years - EXPONENTIAL_START_YEAR))
)

right_final_progress = right_progress[-1]
linear_final_progress = linear_reference_progress[-1]

# ============================================================
# Build DataFrame
# ============================================================

df = pd.DataFrame({
    "year": years,
    "left_progress": left_progress,
    "right_progress": right_progress,
    "linear_reference_progress": linear_reference_progress
})

# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"

left_color = "#ffaa33"
right_color = "#00ff88"
reference_color = "#777777"
quit_line_color = "rgba(255,120,120,0.85)"
growth_start_line_color = "rgba(255,255,255,0.45)"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# Shared y-axis range across both charts
progress_min = min(
    df["left_progress"].min(),
    df["right_progress"].min(),
    df["linear_reference_progress"].min()
)

progress_max = max(
    df["left_progress"].max(),
    df["right_progress"].max(),
    df["linear_reference_progress"].max()
)

y_pad = max((progress_max - progress_min) * 0.12, 5)

Y_RANGE = [
    max(0, progress_min - y_pad),
    progress_max + y_pad
]

# ============================================================
# Figure
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(
        (
            f"{LEFT_PATH_NAME}: "
            f"Shallow Slope → Quit at Year {QUIT_YEAR}"
        ),
        (
            f"{RIGHT_PATH_NAME}: "
            f"Linear Until Year {EXPONENTIAL_START_YEAR} → Exponential Growth"
        )
    )
)

fig.update_annotations(font=dict(color=off_white, size=14))

# ============================================================
# Static reference lines and markers
# ============================================================

# Left: quit year marker
fig.add_shape(
    type="line",
    x0=QUIT_YEAR,
    x1=QUIT_YEAR,
    y0=Y_RANGE[0],
    y1=Y_RANGE[1],
    line=dict(color=quit_line_color, width=1.5, dash="dot"),
    opacity=0.85,
    layer="above",
    row=1,
    col=1
)

fig.add_annotation(
    x=QUIT_YEAR,
    y=Y_RANGE[1],
    text="Quit",
    showarrow=False,
    font=dict(color=off_white, size=12),
    bgcolor="rgba(0,0,0,0.35)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="top",
    row=1,
    col=1
)

# Right: linear reference line
fig.add_trace(
    go.Scatter(
        x=df["year"],
        y=df["linear_reference_progress"],
        mode="lines",
        line=dict(color=reference_color, width=2, dash="dash"),
        name="Linear Reference",
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Linear Reference Progress: %{y:.2f}<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=2
)

# Right: exponential start marker
fig.add_shape(
    type="line",
    x0=EXPONENTIAL_START_YEAR,
    x1=EXPONENTIAL_START_YEAR,
    y0=Y_RANGE[0],
    y1=Y_RANGE[1],
    line=dict(color=growth_start_line_color, width=1.5, dash="dot"),
    opacity=0.85,
    layer="above",
    row=1,
    col=2
)

fig.add_annotation(
    x=EXPONENTIAL_START_YEAR,
    y=Y_RANGE[1],
    text="Exponential starts",
    showarrow=False,
    font=dict(color=off_white, size=12),
    bgcolor="rgba(0,0,0,0.35)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="top",
    row=1,
    col=2
)

# ============================================================
# Initial traces
# ============================================================

# Left: shallow learning curve that stops
fig.add_trace(
    go.Scatter(
        x=[df["year"].iloc[0]],
        y=[df["left_progress"].iloc[0]],
        mode="lines",
        line=dict(color=left_color, width=4),
        name=LEFT_PATH_NAME,
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Progress: %{y:.2f}<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=1
)

# Right: compounding learner path
fig.add_trace(
    go.Scatter(
        x=[df["year"].iloc[0]],
        y=[df["right_progress"].iloc[0]],
        mode="lines",
        line=dict(color=right_color, width=4),
        name=RIGHT_PATH_NAME,
        hovertemplate=(
            "Year: %{x:.1f}<br>"
            "Progress: %{y:.2f}<extra></extra>"
        ),
        showlegend=False
    ),
    row=1,
    col=2
)

# Trace indices
REFERENCE_TRACE_INDEX = 0
LEFT_TRACE_INDEX = 1
RIGHT_TRACE_INDEX = 2

# ============================================================
# In-chart labels
# ============================================================

left_label_x = df["year"].iloc[int(len(df) * 0.05)]
right_label_x = df["year"].iloc[int(len(df) * 0.05)]

label_y_top = Y_RANGE[1] - (Y_RANGE[1] - Y_RANGE[0]) * 0.12
label_y_second = Y_RANGE[1] - (Y_RANGE[1] - Y_RANGE[0]) * 0.25

fig.add_annotation(
    x=left_label_x,
    y=label_y_top,
    text=(
        f"━ {LEFT_PATH_NAME}<br>"
        f"Progress at Quit: {left_quit_progress:.2f}<br>"
        f"Final Progress: {left_final_progress:.2f}"
    ),
    showarrow=False,
    font=dict(color=left_color, size=13),
    align="left",
    bgcolor="rgba(0,0,0,0.38)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="top",
    row=1,
    col=1
)

fig.add_annotation(
    x=right_label_x,
    y=label_y_top,
    text=(
        f"━ {RIGHT_PATH_NAME}<br>"
        f"Linear until Year {EXPONENTIAL_START_YEAR}<br>"
        f"Final Progress: {right_final_progress:.2f}"
    ),
    showarrow=False,
    font=dict(color=right_color, size=13),
    align="left",
    bgcolor="rgba(0,0,0,0.38)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="top",
    row=1,
    col=2
)

fig.add_annotation(
    x=right_label_x,
    y=label_y_second,
    text=(
        f"┄ Linear Reference<br>"
        f"Final Reference: {linear_final_progress:.2f}"
    ),
    showarrow=False,
    font=dict(color=reference_color, size=13),
    align="left",
    bgcolor="rgba(0,0,0,0.38)",
    bordercolor="rgba(255,255,255,0.12)",
    borderwidth=1,
    xanchor="left",
    yanchor="top",
    row=1,
    col=2
)

# ============================================================
# Animation frames
# ============================================================

frames = []
slider_steps = []

frame_indices = list(range(1, len(df), FRAME_STRIDE))

if not frame_indices or frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    plot_slice = df.iloc[:i + 1]

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=plot_slice["year"],
                    y=plot_slice["left_progress"]
                ),
                go.Scatter(
                    x=plot_slice["year"],
                    y=plot_slice["right_progress"]
                )
            ],
            traces=[
                LEFT_TRACE_INDEX,
                RIGHT_TRACE_INDEX
            ],
            name=frame_name
        )
    )

    year_label = df["year"].iloc[i]

    if abs(year_label - round(year_label)) < 0.025:
        slider_label = f"Y{int(round(year_label))}"
    else:
        slider_label = ""

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": slider_label,
        "method": "animate"
    })

fig.frames = frames

# ============================================================
# Layout
# ============================================================

fig.update_layout(
    title=dict(
        text=(
            "Progress Over Time: Quitting Early vs. Staying Long Enough for Compounding"
            "<br><sup>"
            f"Left: shallow early progress leads to quitting at Year {QUIT_YEAR} · "
            f"Right: tracks linear growth first, then compounds after Year {EXPONENTIAL_START_YEAR}"
            "</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=FIG_HEIGHT,
    width=FIG_WIDTH,
    margin=dict(t=120, b=150, r=50, l=75),
    showlegend=False,
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {
                            "duration": FRAME_DURATION,
                            "redraw": False
                        },
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {
                            "duration": 0,
                            "redraw": False
                        },
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Year: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

# ============================================================
# Axes
# ============================================================

fig.update_xaxes(
    axis_style,
    row=1,
    col=1,
    range=[START_YEAR, END_YEAR],
    title_text="Time"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=Y_RANGE,
    title_text="Progress"
)

fig.update_xaxes(
    axis_style,
    row=1,
    col=2,
    range=[START_YEAR, END_YEAR],
    title_text="Time"
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=Y_RANGE,
    title_text="Progress"
)

# ============================================================
# Show / save
# ============================================================

fig.show()

# Optional export:
# fig.write_html("learning_curve_linear_then_exponential_animated.html", include_plotlyjs="cdn")

###### ______________________________________________________________________________________________________________________________________

##### **Reality:** Mastering Difficult Concepts Creates Non-Linear Payoffs

However delayed, these future payoffs often come with convexity which most people don't understand.

Theres no free lunch, or as I say "you don't get somethin for nothin".

If you want to master your quantitative skills, check out Quant Guild - no shortcuts, just a platform to learn math, probability, statistics, and quant finance.

Eveything you need in one place.

---

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary** 
  - This notebook explores the mechanics and psychology behind financial scams, with a focus on why "day trading" and similar promises of easy market success are so compelling—and dangerous—for individuals.
  - We show, through synthetic simulations, how extreme wins and apparent skill can occur purely by chance, making risky trading strategies appear much more effective than they actually are.
  - The main illustration compares a "Day Trading Student" path, which experiences wild gains and catastrophic blowup, to a "Positive EV" (expected value) disciplined strategy—revealing that luck and selective reporting routinely disguise poor risk management as genuine outperformance.
  - Key metrics, such as annualized return, Sharpe ratio, and drawdown, are presented to show that chasing outsized gains with little understanding of risk usually leads to ruin, while patience and positive expectancy steadily compound wealth.
  - The takeaway: It's easy to sell hope and hype in finance, especially when real risk is invisible in short samples—but in the long run, discipline, diversification, and understanding risk are what build sustainable returns.

###### ______________________________________________________________________________________________________________________________________

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$